In [1]:
import tarfile
import zipfile
import io
import os
import time
import math
import pickle
import itertools as itr
import functools as ft
import regex as re
import chardet
from tqdm.auto import tqdm
from pytictoc import TicToc
import json

import pandas as pd
import numpy as np

import gcsfs
fs = gcsfs.GCSFileSystem()

from google.cloud import storage
from google.cloud.exceptions import ClientError

PROJECT_ID = "arxiv-development"
PRD_PROJECT = 'arxiv-production'
PRD_BUCKET_LOC = 'arxiv-production-data' 

from pylatexenc.latexwalker import LatexWalker, LatexEnvironmentNode, LatexGroupNode, LatexMacroNode, LatexCharsNode
from pylatexenc.latex2text import LatexNodes2Text


In [2]:
os.chdir("/home/jupyter/metadata-vertexai/")  # this needs to be the folder where notebook lives
import importlib
import phase_one_json as phase_one


  0%|          | 0/2091 [00:00<?, ?it/s]

  0%|          | 0/44 [00:00<?, ?it/s]

In [3]:
from IPython.core.interactiveshell import InteractiveShell
# pretty print all cell's output and not just the last one
InteractiveShell.ast_node_interactivity = "all"

In [4]:
def safe_divide(num, denom):
    return num / denom if denom != 0 else 0.0

Note that id lists were prepared previously from the DB, using:  

```
select concat(paper_id,"v",version) as arx_id from arXiv_metadata
where paper_id LIKE "23%";
and is_withdrawn != 1
and is_current = 1;
```

In [5]:
ror_gspath = 'gs://institutional-extract-scratch/reference/v1.63-2025-04-03-ror-data_schema_v2.json'
fs = gcsfs.GCSFileSystem()
with fs.open(ror_gspath, "r", encoding="utf-8") as f:
    ror_data = json.load(f)
ror_dict = {x['id'].rsplit('/')[-1]: x for x in ror_data}

In [6]:
#ror_dict['043mz5j54']

In [7]:
def get_children_for_ror(target_id):
    res_list = []
    rels = ror_dict[target_id].get('relationships',[])
    for rel in rels:
        rel_type = rel.get('type','')
        if rel_type != 'child':
            continue
        child_id = rel.get('id','').rsplit('/')[-1]
        if child_id:
            res_list.append(child_id)
    return res_list


## Set parameters

In [8]:
#results_file = "gs://institutional-extract-scratch/reference/arx_ids/2311_ids.csv"
#results_file = "gs://institutional-extract-scratch/output/2311_db_all_2025-04-28.csv.zip"
results_file = "gs://institutional-extract-scratch/output/2311_db_json_all_2025-05-01.csv.zip"


In [9]:
test_ids_df = pd.read_csv(results_file)

test_ids_df.head()
ids_2311_all = test_ids_df["arx_id"].unique()

,arx_id,name,ror
0,2311.06393v1,"Indian Institute of Technology Guwahati, Guwahati",0022nd079
1,2311.06393v1,"Indian Institute of Technology Guwahati, Guwahati",0022nd079
2,2311.17394v1,"Nanyang Technological University, Singapore",02e7b5302
3,2311.00854v2,"University of Ioannina, Greece",01qg3j183
4,2311.11676v1,"Southwest University, Chongqing",01kj4z117


In [10]:
test_ids_df.shape

(52181, 3)

In [11]:
vip_df = pd.read_csv("gs://institutional-extract-scratch/reference/dashboard_institutions2024_2025-05-01.csv", dtype=str)

vip_df = vip_df[vip_df['is_consortium']=='0']
vip_df.shape
vip_df.head()

(345, 21)

,sid,name,country,country_code,consortia_code,member_type,ror_id,sub_rors,is_consortium,label,...,is_active,Institution,salsaId,orgId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
0,447,Aalto University,Finland,FI,FinELib,member,https://ror.org/020hwjq30,NaN,0,Aalto University,...,1,Aalto University,52318446,60103653,NaN,antti.m.rousi@aalto.fi,Antti,Rousi,"Specialist, Research Services",NaN
1,465,Abo Akademi University,Finland,FI,FinELib,member,https://ror.org/029pk6x14,NaN,0,Abo Akademi University,...,1,Abo Akademi University,NaN,60015375,No contact for school this is the consortium c...,timo.vilen@helsinki.fi,Timo,Vilén,Information Specialist,NaN
2,482,Ames Laboratory,United States,US,NaN,member,https://ror.org/041m9xr71,NaN,0,Ames Laboratory,...,1,Ames Laboratory,52320619,60008023,NaN,lgraves@iastate.edu,Laura,Graves,NaN,NaN
3,483,Argonne National Lab,United States,US,NaN,member,https://ror.org/05gvnxz63,NaN,0,Argonne National Lab,...,1,Argonne National Lab,1714199,60028609,NaN,mstraka@anl.gov,Mary,Straka,NaN,NaN
4,16,Australian National University,Australia,AU,CAUL,member,https://ror.org/019wvm592,NaN,0,Australian National University,...,1,Australian National University,2090464,60008950,NaN,electronic.coordinator@anu.edu.au,NaN,NaN,NaN,NaN


In [12]:
missing_vip = {
    '052rrw050': "National Astronomical Observatory of Japan",
    '049bh0z35': "National Library of Sweden",
    '028rypz17': "Université Paris-Sud",    
    '021f7p178': "Lib4RI",
    '006gw6z14': "CSIC- Estación Biológica de Doñana EBD",
    '04zp24820': "Chennai Mathematical Institute",
    '04nrv3s86': "CSIC-UMA - Instituto de Hortofruticultura Subtropical y Mediterranea La Mayora (IHSM)",
    '03srn9y98': "CSIC - Instituto de Química Avanzada de Cataluña (IQAC)",
    '03kgj4539': "TRIUMF",
    '000nhpy59': "CSIC-UMH - Instituto de Neurociencias (IN)",
    '00bwtjf83': "Tampere University of Applied Sciences",
}

vip_dna_df = vip_df.dropna(subset=['ror_id'])
for ror in missing_vip:
    idx = vip_dna_df['ror_id'].str.endswith(ror)
    vip_dna_df.loc[idx,['orgId', 'name', 'country', 'ror_id']]

,orgId,name,country,ror_id
168,60029701,National Astronomical Observatory of Japan,Japan,https://ror.org/052rrw050


,orgId,name,country,ror_id
169,NaN,National Library of Sweden,Sweden,https://ror.org/049bh0z35


,orgId,name,country,ror_id
334,NaN,Université Paris-Sud,France,https://ror.org/028rypz17


,orgId,name,country,ror_id
152,NaN,Lib4RI,Switzerland,https://ror.org/021f7p178


,orgId,name,country,ror_id
51,60006627,CSIC- Estación Biológica de Doñana EBD,Spain,https://ror.org/006gw6z14


...

,orgId,name,country,ror_id
64,60104354,CSIC-UMA - Instituto de Hortofruticultura Subt...,Spain,https://ror.org/04nrv3s86


,orgId,name,country,ror_id
44,60103742,CSIC - Instituto de Química Avanzada de Catalu...,Spain,https://ror.org/03srn9y98


,orgId,name,country,ror_id
206,60000731,TRIUMF,Canada,https://ror.org/03kgj4539


,orgId,name,country,ror_id
65,60017107,CSIC-UMH - Instituto de Neurociencias (IN),Spain,https://ror.org/000nhpy59


,orgId,name,country,ror_id
208,60110687,Tampere University of Applied Sciences,Finland,https://ror.org/00bwtjf83


In [13]:
missing_vip = {
    '04zp24820': "Chennai Mathematical Institute",
    '021f7p178': "Lib4RI",
    '052rrw050': "National Astronomical Observatory of Japan",
    '049bh0z35': "National Library of Sweden",
    '03kgj4539': "TRIUMF",
    '00bwtjf83': "Tampere University of Applied Sciences",
    '028rypz17': "Université Paris-Sud",
}
vip_dna_df = vip_df.dropna(subset=['ror_id'])
for ror in missing_vip:
    idx = vip_dna_df['ror_id'].str.endswith(ror)
    vip_dna_df.loc[idx,['orgId', 'name', 'country', 'ror_id']]

,orgId,name,country,ror_id
86,60019021,Chennai Mathematical Institute,India,https://ror.org/04zp24820


,orgId,name,country,ror_id
152,NaN,Lib4RI,Switzerland,https://ror.org/021f7p178


,orgId,name,country,ror_id
168,60029701,National Astronomical Observatory of Japan,Japan,https://ror.org/052rrw050


,orgId,name,country,ror_id
169,NaN,National Library of Sweden,Sweden,https://ror.org/049bh0z35


,orgId,name,country,ror_id
206,60000731,TRIUMF,Canada,https://ror.org/03kgj4539


,orgId,name,country,ror_id
208,60110687,Tampere University of Applied Sciences,Finland,https://ror.org/00bwtjf83


,orgId,name,country,ror_id
334,NaN,Université Paris-Sud,France,https://ror.org/028rypz17


In [14]:
vip_df['ror_id'].value_counts()
vip_df['orgId'].value_counts()

ror_id
https://ror.org/04tsk2644    2
https://ror.org/03nawhv43    1
https://ror.org/00d9ah105    1
https://ror.org/046rm7j60    1
https://ror.org/04gyf1771    1
                            ..
https://ror.org/041kmwe10    1
https://ror.org/03gnh5541    1
https://ror.org/03v8tnc06    1
https://ror.org/01hcx6992    1
https://ror.org/04ka0vh05    1
Name: count, Length: 338, dtype: int64

orgId
60005322    2
60029526    1
60027550    1
60007278    1
60014439    1
           ..
60117390    1
60000762    1
60014652    1
60030788    1
60032005    1
Name: count, Length: 339, dtype: int64

In [15]:
vip_df[vip_df['orgId'] == "60015150"]
vip_df[vip_df['name'].str.contains("Paris")]

,sid,name,country,country_code,consortia_code,member_type,ror_id,sub_rors,is_consortium,label,...,is_active,Institution,salsaId,orgId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
122,570,"Imperial College of Science, Technology, and M...",United Kingdom,GB,NaN,champion,https://ror.org/041kmwe10,NaN,0,"Imperial College of Science, Technology, and M...",...,1,"Imperial College of Science, Technology, and M...",NaN,60015150,NaN,"libsubs@imperial.ac.uk,robyn.price@imperial.ac.uk",NaN,NaN,NaN,NaN


,sid,name,country,country_code,consortia_code,member_type,ror_id,sub_rors,is_consortium,label,...,is_active,Institution,salsaId,orgId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
7,235,Bibliothèque de l'Observatoire de Paris (OBSPM),France,FR,CCSD,member,https://ror.org/029nkcm90,NaN,0,Bibliothèque de l'Observatoire de Paris (OBSPM),...,1,Bibliothèque de l'Observatoire de Paris (OBSPM),NaN,60003674,NaN,NaN,NaN,NaN,NaN,NaN
181,596,Paris Nanterre University,France,FR,COUP,member,https://ror.org/013bkhk48,NaN,0,Paris Nanterre University,...,1,Paris Nanterre University,NaN,60017338,NaN,mleguenn@parisnanterre.fr,NaN,NaN,NaN,NaN
333,653,Université Paris-Saclay,France,FR,CCSD,member,https://ror.org/03xjwb503,028rypz17,0,Université Paris-Saclay,...,1,Université Paris-Saclay,NaN,60106017,NaN,NaN,NaN,NaN,NaN,NaN
334,654,Université Paris-Sud,France,FR,CCSD,member,https://ror.org/028rypz17,03xjwb503),0,Université Paris-Sud,...,1,Université Paris-Sud,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
vip_df.head()['ror_id'].apply(lambda x: x.rsplit('/')[-1])

0    020hwjq30
1    029pk6x14
2    041m9xr71
3    05gvnxz63
4    019wvm592
Name: ror_id, dtype: object

In [17]:
vip_df.head()

,sid,name,country,country_code,consortia_code,member_type,ror_id,sub_rors,is_consortium,label,...,is_active,Institution,salsaId,orgId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
0,447,Aalto University,Finland,FI,FinELib,member,https://ror.org/020hwjq30,NaN,0,Aalto University,...,1,Aalto University,52318446,60103653,NaN,antti.m.rousi@aalto.fi,Antti,Rousi,"Specialist, Research Services",NaN
1,465,Abo Akademi University,Finland,FI,FinELib,member,https://ror.org/029pk6x14,NaN,0,Abo Akademi University,...,1,Abo Akademi University,NaN,60015375,No contact for school this is the consortium c...,timo.vilen@helsinki.fi,Timo,Vilén,Information Specialist,NaN
2,482,Ames Laboratory,United States,US,NaN,member,https://ror.org/041m9xr71,NaN,0,Ames Laboratory,...,1,Ames Laboratory,52320619,60008023,NaN,lgraves@iastate.edu,Laura,Graves,NaN,NaN
3,483,Argonne National Lab,United States,US,NaN,member,https://ror.org/05gvnxz63,NaN,0,Argonne National Lab,...,1,Argonne National Lab,1714199,60028609,NaN,mstraka@anl.gov,Mary,Straka,NaN,NaN
4,16,Australian National University,Australia,AU,CAUL,member,https://ror.org/019wvm592,NaN,0,Australian National University,...,1,Australian National University,2090464,60008950,NaN,electronic.coordinator@anu.edu.au,NaN,NaN,NaN,NaN


### Eval results

In [75]:
ror_wd_map = phase_one.ROR_FINDER.withdrawn_map
ror_wd_map.get('https://ror.org/04zrf7b53')
ror_wd_map.get('https://ror.org/02aj0kh94')

'https://ror.org/01qrts582'

'https://ror.org/02rx3b187'

In [44]:
vip_df = pd.read_csv("gs://institutional-extract-scratch/reference/dashboard_institutions2024_2025-05-01.csv", dtype=str)
vip_df = vip_df[vip_df['is_consortium']=='0']
# remap inactive/withdrawn to successors
vip_df['ror_id'] = vip_df['ror_id'].replace(ror_wd_map)


In [45]:
ror_df = pd.DataFrame({
    'name':vip_df['name'],
    'ror':vip_df['ror_id'].apply(lambda x: pd.NA if pd.isna(x) else x.rsplit('/')[-1])
})
ror_df = ror_df.dropna()

In [46]:
ror_map_df = pd.read_csv("gs://institutional-extract-scratch/reference/matched_results_ror_api.csv", dtype=str)
ror_map_df = ror_map_df.set_index('Primary Org Id')
# remap inactive/withdrawn to successors
ror_map_df['ROR ID'] = ror_map_df['ROR ID'].replace(ror_wd_map)
ror_map_df.head()
ror_map_df[ror_map_df['ROR ID'] == 'https://ror.org/01qrts582'].head(30)

,Primary Org Name,Country Name,ROR ID
Primary Org Id,,,
60000009,Villanova University,United States,https://ror.org/02g7kd627
60000011,Saitama Institute of Technology,Japan,https://ror.org/01pkeax38
60000015,Lusófona University,Portugal,https://ror.org/05xxfer42
60000021,Atatürk Üniversitesi,Turkey,https://ror.org/03je5c526
60000027,KLA Corporation,United States,https://ror.org/04zdyxh40


,Primary Org Name,Country Name,ROR ID
Primary Org Id,,,
60009941,Technische Universität Kaiserslautern,Germany,https://ror.org/01qrts582


In [47]:
vip_df = vip_df.set_index('orgId')
ror_map_df = pd.merge(
    ror_map_df, 
    vip_df[['ror_id']],
    how='outer', left_index=True, right_index=True,
)
ror_map_df['ror_id'] = ror_map_df['ror_id'].fillna(ror_map_df['ROR ID'])
ror_map_df.drop(columns=['ROR ID'], inplace=True)
ror_map_df = ror_map_df.reset_index()
ror_map_df = ror_map_df.dropna(subset=['Primary Org Id', 'Primary Org Name', 'ror_id']).drop_duplicates()
ror_map_df['ror'] = ror_map_df['ror_id'].apply(lambda x: pd.NA if pd.isna(x) else x.rsplit('/')[-1])
rors_in_map = set(ror_map_df['ror'].unique())
ror_map_df.shape
ror_map_df.head()

(13236, 5)

,Primary Org Id,Primary Org Name,Country Name,ror_id,ror
0,60000009,Villanova University,United States,https://ror.org/02g7kd627,02g7kd627
1,60000011,Saitama Institute of Technology,Japan,https://ror.org/01pkeax38,01pkeax38
2,60000015,Lusófona University,Portugal,https://ror.org/05xxfer42,05xxfer42
3,60000021,Atatürk Üniversitesi,Turkey,https://ror.org/03je5c526,03je5c526
4,60000027,KLA Corporation,United States,https://ror.org/04zdyxh40,04zdyxh40


In [48]:
ror_map_df[ror_map_df['Primary Org Id'] == '60005322']
ror_map_df.value_counts("Primary Org Id")
ror_map_df.isna().sum()

,Primary Org Id,Primary Org Name,Country Name,ror_id,ror
1018,60005322,Ruhr-Universitat Bochum,Germany,https://ror.org/04tsk2644,04tsk2644


Primary Org Id
60279976    1
60000009    1
60000011    1
60000015    1
60279685    1
           ..
60000060    1
60000056    1
60000050    1
60000036    1
60000027    1
Name: count, Length: 13236, dtype: int64

Primary Org Id      0
Primary Org Name    0
Country Name        0
ror_id              0
ror                 0
dtype: int64

In [49]:
scopus_df = pd.read_csv("gs://institutional-extract-scratch/training/2311_scopus_17416.csv.zip", dtype=str)
scopus_all = set(scopus_df["ArXiv Id"].unique())
scopus_positive = scopus_df[scopus_df['Primary Org Id'] == 60027550]["ArXiv Id"].unique()

scopus_df = pd.merge(
    scopus_df, 
    ror_map_df[["Primary Org Id", "ror"]], 
    how='left',
    on='Primary Org Id'
)
scopus_all = set(scopus_df["paper_id"].unique())

In [50]:
scopus_df['paper_id'].nunique()
scopus_df.shape
scopus_df.head()

17416

(80364, 9)

,Primary Org Id,Primary Org Name,Primary Org City,Primary Org State,Primary Org Country,ArXiv Id,Affiliation Sequence Number,paper_id,ror
0,60006297,University of Pennsylvania,Philadelphia,PA,United States,2311.03477v1,1,2311.03477,00b30xv10
1,60024190,"Institute of Plasma Physics, Academy of Scienc...",Prague,NaN,Czech Republic,2311.04187v1,9,2311.04187,01h494015
2,60025641,Universität Freiburg,Freiburg im Breisgau,Baden-Wurttemberg,Germany,2311.04557v1,1,2311.04557,0245cg223
3,60114755,"Istituto Nazionale di Fisica Nucleare, Sezione...",Milan,NaN,Italy,2311.14088v1,27,2311.14088,04w4m6z96
4,60115855,Trento Institute for Fundamental Physics and A...,Povo,TN,Italy,2311.09750v1,2,2311.09750,00nhs3j29


In [51]:
afile_df = pd.read_csv( 'stlouis.csv', dtype=str, index_col=None, usecols=['arx_id','name','ror'])
afile_df.head()
afile_df.isnull().sum()
afile_df[afile_df['arx_id']=='2311.00048v2']

,arx_id,name,ror
0,2311.05497v2,IXPE Collaboration,NaN
1,2311.05497v2,Newcastle University,01kj2bm70
2,2311.05497v2,University of Turku,05vghhr25
3,2311.05497v2,KTH Royal Institute of Technology,026vcq606
4,2311.05497v2,Stockholm University,05f0yaq80


arx_id     0
name       0
ror       27
dtype: int64

,arx_id,name,ror
582,2311.00048v2,Mallinckrodt Institute of Radiology,NaN
583,2311.00048v2,Washington University School of Medicine,01yc7t268
584,2311.00048v2,Arizona State University,03efmqc40


In [52]:
res_df = pd.read_csv(results_file)
append_files = [
    #"paris-sud.csv",
    #"king-college.csv",
    'stlouis.csv',
    'stlouis2.csv',
]
for afile in append_files:
    afile_df = pd.read_csv(afile, dtype=str, index_col=None, usecols=['arx_id', 'name', 'ror'])
    res_df = pd.concat([res_df, afile_df], axis=0).reset_index(drop=True)
res_df.shape

res_df['paper_id'] = res_df['arx_id'].apply(lambda x: x.rsplit('v',1)[0])
res_df.shape
res_df.head()

res_df[res_df['arx_id']=='2311.03508v2']

(53960, 3)

(53960, 4)

,arx_id,name,ror,paper_id
0,2311.06393v1,"Indian Institute of Technology Guwahati, Guwahati",0022nd079,2311.06393
1,2311.06393v1,"Indian Institute of Technology Guwahati, Guwahati",0022nd079,2311.06393
2,2311.17394v1,"Nanyang Technological University, Singapore",02e7b5302,2311.17394
3,2311.00854v2,"University of Ioannina, Greece",01qg3j183,2311.00854
4,2311.11676v1,"Southwest University, Chongqing",01kj4z117,2311.11676


,arx_id,name,ror,paper_id
527,2311.03508v2,"Louis, St. Louis",0395sep02,2311.03508
528,2311.03508v2,"University of California at Riverside, Riverside",03nawhv43,2311.03508
529,2311.03508v2,"Louis, St. Louis",0395sep02,2311.03508
52412,2311.03508v2,Washington University in St. Louis,01yc7t268,2311.03508
52413,2311.03508v2,University of California at Riverside,03nawhv43,2311.03508


In [53]:
skip_inst = {
    '01nsd7y51': 'CSIC - Geociencias Barcelona (GEO3BCN)',
    '05dsysc59': 'CSIC - Instituto de Carboquímica (ICB)',
    '04zdays56': 'CSIC - Instituto de Biologia Molecular y Celul...',
    '03hasqf61': 'CSIC - Institut de Ciència de Materials de Bar...',
    '02h7vfp25': 'CSIC - Instituto de Cerámica y Vidrio (ICV)',
    '04qayn356': 'CSIC - Instituto de Ciencias Marinas de Andalu...',
}

scopus_to_rerun = set()
vip_results = []
for ror in tqdm(ror_df['ror'].unique()):
    if ror in skip_inst:
        continue
    if not ror in rors_in_map:
        print(f"'{ror}': {ror_df.loc[ror_df['ror']==ror,'name'].iloc[0]}")
        continue
    inst_name = ror_map_df[ror_map_df['ror']==ror]['Primary Org Name'].iloc[0]
    sub_ids = get_children_for_ror(ror)
    sub_ids.append(ror)
    scopus_ids = set(scopus_df[scopus_df['ror'].isin(sub_ids)]['paper_id'].unique())
    scopus_arx_ids = set(scopus_df[scopus_df['ror'].isin(sub_ids)]['ArXiv Id'].unique())
    scopus_to_rerun = scopus_to_rerun.union(scopus_arx_ids)
    res_ids = set(res_df[res_df['ror'].isin(sub_ids)]['arx_id'].apply(lambda x: x.rsplit('v')[0]).unique())
    res_ids = res_ids.intersection(scopus_all) # only consider cases in the scopus data
    res = {
        "name": inst_name,
        "ror": ror,
        "TP": len(res_ids.intersection(scopus_ids)), 
        "FP": len(res_ids - scopus_ids), 
        "FN": len(scopus_ids - res_ids), 
        "TN": len((scopus_all - scopus_ids) - res_ids)
    }
    res['precision'] = safe_divide(res['TP'], res['TP'] + res['FP'])
    res['recall'] = safe_divide(res['TP'], res['TP'] + res['FN'])
    if (res['precision'] + res['recall']) > 0:
        res['F1'] = safe_divide(2 * res['precision'] * res['recall'], res['precision'] + res['recall'])
    else:
        res['F1'] = 0.0
    vip_results.append(res)


  0%|          | 0/337 [00:00<?, ?it/s]

'03srn9y98': CSIC - Instituto de Química Avanzada de Cataluña (IQAC)
'006gw6z14': CSIC- Estación Biológica de Doñana EBD
'04nrv3s86': CSIC-UMA - Instituto de Hortofruticultura Subtropical y Mediterranea La Mayora (IHSM)
'000nhpy59': CSIC-UMH - Instituto de Neurociencias (IN)
'04zp24820': Chennai Mathematical Institute
'021f7p178': Lib4RI
'052rrw050': National Astronomical Observatory of Japan
'049bh0z35': National Library of Sweden
'03kgj4539': TRIUMF
'00bwtjf83': Tampere University of Applied Sciences


In [54]:
vip_res_df = pd.DataFrame.from_records(vip_results)
vip_res_df.head()
vip_res_df.describe()

,name,ror,TP,FP,FN,TN,precision,recall,F1
0,Aalto University,020hwjq30,62,0,19,17335,1.000000,0.765432,0.867133
1,Åbo Akademi University,029pk6x14,2,0,0,17414,1.000000,1.000000,1.000000
2,Ames Laboratory,041m9xr71,4,0,2,17410,1.000000,0.666667,0.800000
3,Argonne National Laboratory,05gvnxz63,42,3,15,17356,0.933333,0.736842,0.823529
4,The Australian National University,019wvm592,72,3,6,17335,0.960000,0.923077,0.941176


,TP,FP,FN,TN,precision,recall,F1
count,321.000000,321.000000,321.000000,321.000000,321.000000,321.000000,321.000000
mean,47.218069,2.909657,12.317757,17353.554517,0.824257,0.689415,0.739388
std,56.330463,5.644793,19.191110,71.942192,0.325270,0.310862,0.310829
min,0.000000,0.000000,0.000000,16938.000000,0.000000,0.000000,0.000000
25%,9.000000,0.000000,1.000000,17332.000000,0.894737,0.615385,0.735849
50%,31.000000,1.000000,8.000000,17375.000000,0.960000,0.806452,0.870588
75%,65.000000,3.000000,16.000000,17405.000000,1.000000,0.905263,0.929134
max,393.000000,58.000000,173.000000,17416.000000,1.000000,1.000000,1.000000


In [55]:
len(scopus_to_rerun)


9736

In [56]:
#vip_res_df.to_csv("vip_results_2025-05-01.csv") 

In [57]:
total_gt_cases = vip_res_df[['TP', 'FP', 'FN', 'TN']].iloc[0].sum()

In [58]:
cutoff = 10
tn_min = total_gt_cases - cutoff
vip_res_df.query("TN <= @tn_min").sort_values('F1', ascending=True).head(20)
vip_res_df[vip_res_df['ror'].str.contains('03xjwb503')].sort_values('F1', ascending=True).head(10)

,name,ror,TP,FP,FN,TN,precision,recall,F1
44,CSIC-UAM - Instituto de Física Teórica (IFT),022r8mj40,3,2,30,17381,0.600000,0.090909,0.157895
69,Commissariat a l'Energie Atomique et aux Energ...,00jjx8s55,42,4,173,17197,0.913043,0.195349,0.321839
135,Memorial University of Newfoundland,04haebc03,2,0,8,17406,1.000000,0.200000,0.333333
59,CSIC-UV - Instituto de Física Corpuscular,017xch102,7,1,25,17383,0.875000,0.218750,0.350000
143,Foundation for Fundamental Research on Matter,00f9tz983,12,1,33,17370,0.923077,0.266667,0.413793
47,CSIC-UC - Instituto de Física de Cantabria (IFCA),040kx1j83,3,0,7,17406,1.000000,0.300000,0.461538
299,Université Toulouse III - Paul Sabatier,02v6kpv12,30,3,63,17320,0.909091,0.322581,0.476190
103,Institute of Physics of the Czech Academy of S...,02yhj4v17,11,0,24,17381,1.000000,0.314286,0.478261
300,University of Fribourg,022fs9h90,6,6,7,17397,0.500000,0.461538,0.480000
301,Université Grenoble Alpes,02rx3b187,47,3,92,17274,0.940000,0.338129,0.497354


,name,ror,TP,FP,FN,TN,precision,recall,F1
298,Université Paris-Saclay,03xjwb503,164,4,150,17098,0.97619,0.522293,0.680498


In [59]:
scopus_df.head()

,Primary Org Id,Primary Org Name,Primary Org City,Primary Org State,Primary Org Country,ArXiv Id,Affiliation Sequence Number,paper_id,ror
0,60006297,University of Pennsylvania,Philadelphia,PA,United States,2311.03477v1,1,2311.03477,00b30xv10
1,60024190,"Institute of Plasma Physics, Academy of Scienc...",Prague,NaN,Czech Republic,2311.04187v1,9,2311.04187,01h494015
2,60025641,Universität Freiburg,Freiburg im Breisgau,Baden-Wurttemberg,Germany,2311.04557v1,1,2311.04557,0245cg223
3,60114755,"Istituto Nazionale di Fisica Nucleare, Sezione...",Milan,NaN,Italy,2311.14088v1,27,2311.14088,04w4m6z96
4,60115855,Trento Institute for Fundamental Physics and A...,Povo,TN,Italy,2311.09750v1,2,2311.09750,00nhs3j29


### Check specific inst

In [61]:
name_txt = "Université Grenoble Alpes"
scopus_df[scopus_df['Primary Org Name'].str.contains(name_txt)].head()
ror_map_df[ror_map_df['Primary Org Name'].str.contains(name_txt)]
vip_df[vip_df['name'].str.contains(name_txt)]
vip_df.loc['60104653']

,Primary Org Id,Primary Org Name,Primary Org City,Primary Org State,Primary Org Country,ArXiv Id,Affiliation Sequence Number,paper_id,ror
553,60104653,Université Grenoble Alpes,Saint Martin d'Heres,Auvergne-Rhone-Alpes,France,2311.12964v1,1,2311.12964,02rx3b187
559,60104653,Université Grenoble Alpes,Saint Martin d'Heres,Auvergne-Rhone-Alpes,France,2311.15371v1,3,2311.15371,02rx3b187
564,60104653,Université Grenoble Alpes,Saint Martin d'Heres,Auvergne-Rhone-Alpes,France,2311.03272v1,6,2311.03272,02rx3b187
2313,60104653,Université Grenoble Alpes,Saint Martin d'Heres,Auvergne-Rhone-Alpes,France,2311.02028v2,4,2311.02028,02rx3b187
2314,60104653,Université Grenoble Alpes,Saint Martin d'Heres,Auvergne-Rhone-Alpes,France,2311.02246v1,1,2311.02246,02rx3b187


,Primary Org Id,Primary Org Name,Country Name,ror_id,ror
9253,60104653,Université Grenoble Alpes,France,https://ror.org/02rx3b187,02rx3b187


,sid,name,country,country_code,consortia_code,member_type,ror_id,sub_rors,is_consortium,label,comment,is_active,Institution,salsaId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
orgId,,,,,,,,,,,,,,,,,,,,


sid                                             656
name                   Université de Grenoble Alpes
country                                      France
country_code                                     FR
consortia_code                                 CCSD
member_type                                  member
ror_id                    https://ror.org/02rx3b187
sub_rors                                        NaN
is_consortium                                     0
label                  Université de Grenoble Alpes
comment                                         NaN
is_active                                         1
Institution            Université de Grenoble Alpes
salsaId                                         NaN
Comment                                         NaN
Usage Contact Email                             NaN
First Name                                      NaN
Last Name                                       NaN
Title                                           NaN
Unnamed: 17 

In [62]:
ror = "02rx3b187"
sub_ids = get_children_for_ror(ror)
sub_ids.append(ror)
print(sub_ids)
scopus_ids = set(scopus_df[scopus_df['ror'].isin(sub_ids)]['paper_id'].unique())
scopus_arx_ids = set(scopus_df[scopus_df['ror'].isin(sub_ids)]['ArXiv Id'].unique())
len(scopus_ids)
res_df[res_df['ror']==ror].shape
res_df[res_df['ror']==ror].head()

['000tdrn36', '00fwjkb59', '05sbt2524', '04dbzz632', '01cf2sz15', '01wwcfa26', '03eqm6y13', '023n9q531', '05588ks88', '04qz4qy85', '03f0apy98', '02rmwrd87', '047p7mf25', '03vte9x46', '05c99vk74', '04fhvpc68', '03jrb0276', '01kbr1737', '00ndvqf03', '0467x8h16', '02cmt9z73', '026m44z54', '03949e763', '04ndt7n58', '044cfnj78', '0509qp208', '03e044190', '01w1erp60', '01c8rcg82', '02wrme198', '04ett5b41', '026j45x50', '04eg25g76', '014p6mg26', '055q9jt53', '05hz99a17', '04e2ndp15', '05bw7ad85', '01a0ez112', '00genbz89', '033d95m27', '00t9pwh21', '02wrd4e19', '00y523k32', '01554y636', '00awrf758', '00rv5x925', '05rwrfh97', '000063q30', '03985kf35', '04as3rk94', '01aj0bf86', '039j4x695', '01027m165', '0557vhy43', '04szabx38', '0003ege03', '010rs2a38', '0459x4g44', '03vyv3y87', '05kwbf598', '02dd25k08', '03x1z2w73', '00jxe7243', '01273vs09', '05514hp74', '0535cbn94', '03t9s7t87', '044ggyg50', '045ktmd28', '02z8yps18', '01kcrnc96', '043pfpy19', '03bcdsr62', '04axb9j69', '014n97s28', '02mc6qk71'

139

(39, 4)

,arx_id,name,ror,paper_id
111,2311.04504v1,"Université Grenoble Alpes, Grenoble",02rx3b187,2311.04504
1506,2311.05002v1,"Grenoble Alpes University, Grenoble",02rx3b187,2311.05002
3213,2311.05948v1,"Université Grenoble Alpes, St Martin d'Hères",02rx3b187,2311.05948
3462,2311.03264v2,"Université Grenoble Alpes, Grenoble",02rx3b187,2311.03264
5988,2311.10535v2,"Université Grenoble Alpes, Grenoble",02rx3b187,2311.10535


In [63]:
#pd.DataFrame({'arx_id':list(scopus_arx_ids)}).to_csv("paris-saclay_arx_ids.csv", header=False)

In [64]:
named_inst = res_df[res_df['paper_id'].isin(scopus_ids)].value_counts('name', ascending=False)
named_inst.head()
res_df[res_df['paper_id'].isin(scopus_ids)].groupby(['name'])['ror'].value_counts(ascending=False).sort_values(ascending=False).head(20)
res_df[res_df['paper_id'].isin(scopus_ids)].head(10)
res_df[res_df['paper_id'].isin(scopus_ids) & res_df['name'].str.contains("Toulouse")].head(10)

name
Grenoble Alpes, Grenoble                                      55
Université Grenoble Alpes, Grenoble                           15
NASA Marshall Space Flight Center, Huntsville                 12
Universidade de Lisboa, Lisboa                                12
INAF Istituto di Astrofisica e Planetologia Spaziali, Roma    11
Name: count, dtype: int64

name                                                          ror      
Grenoble Alpes, Grenoble                                      02tfmen73    55
Université Grenoble Alpes, Grenoble                           02rx3b187    15
NASA Marshall Space Flight Center, Huntsville                 02epydz83    12
Universidade de Lisboa, Lisboa                                01c27hj86    12
University of Helsinki, Helsinki                              040af2s02    11
Universidade do Porto, Porto                                  043pwc612    11
INAF Istituto di Astrofisica e Planetologia Spaziali, Roma    0141xw169    11
Universidad Polit\'ecnica de Cartagena, Cartagena             02k5kx966    10
Agenzia Spaziale Italiana, Roma                               034zgem50    10
California Institute of Technology, Pasadena                  05dxps055     9
CNRS, Grenoble                                                02feahw73     9
Leiden University, Leiden                                     027bh9e2

,arx_id,name,ror,paper_id
111,2311.04504v1,"Université Grenoble Alpes, Grenoble",02rx3b187,2311.04504
112,2311.04504v1,"CNRS, Toulouse",02feahw73,2311.04504
113,2311.04504v1,"Institut de Ciència de Materials de Barcelona,...",03hasqf61,2311.04504
396,2311.17659v3,"Universit\""at Bayreuth, Bayreuth",0234wmv40,2311.17659
397,2311.17659v3,"Grenoble Alpes, Grenoble",02tfmen73,2311.17659
881,2311.03272v1,"Universität Wien, Vienna",03prydq77,2311.03272
882,2311.03272v1,"Konkoly Observatory, Budapest",039jmcx36,2311.03272
883,2311.03272v1,"University College London, London",02jx3x895,2311.03272
884,2311.03272v1,"Ghent University, Gent",00cv9y106,2311.03272
885,2311.03272v1,"Grenoble Alpes, Grenoble",02tfmen73,2311.03272


,arx_id,name,ror,paper_id
112,2311.04504v1,"CNRS, Toulouse",02feahw73,2311.04504
10276,2311.13513v1,"de Toulouse, Toulouse, France",017h5q109,2311.13513
11003,2311.13625v1,"Université de Toulouse, Toulouse",017tgbk05,2311.13625
13194,2311.08307v1,"Universit\'{e} de Toulouse, Toulouse",004raaa70,2311.08307
22496,2311.07011v2,"Universit\'e de Toulouse, Toulouse",004raaa70,2311.07011
22736,2311.05216v1,Centre d'Elaboration de Matériaux et Etudes St...,03kwnqq69,2311.05216
25620,2311.03168v1,"Université de Toulouse, Toulouse",017tgbk05,2311.03168
29747,2311.12096v2,"Centre National d'Etudes Spatiales, Toulouse",04h1h0y33,2311.12096
29759,2311.12096v2,"Université de Toulouse, Toulouse",017tgbk05,2311.12096
30215,2311.06922v4,"LAAS-CNRS, Toulouse",03vcm6439,2311.06922


#### Target Missing

In [65]:
paper_grps = res_df[res_df['paper_id'].isin(scopus_ids)].groupby(['arx_id'])['ror'].unique().reset_index()
sub_ids[:10]
error_df = paper_grps[paper_grps['ror'].apply(lambda x: all(i not in sub_ids for i in x))]
error_df.head(10)
check_ids = error_df['arx_id'].unique().tolist()
print(f"Papers missing target {len(check_ids)} of {len(scopus_arx_ids)}")

['000tdrn36',
 '00fwjkb59',
 '05sbt2524',
 '04dbzz632',
 '01cf2sz15',
 '01wwcfa26',
 '03eqm6y13',
 '023n9q531',
 '05588ks88',
 '04qz4qy85']

,arx_id,ror
0,2311.00132v1,"[02aj0kh94, 04teye511, 047gc3g35]"
4,2311.01250v2,"[036x5ad56, 00240q980, 05v727m31, 0220mzb33, 0..."
5,2311.01262v2,"[02tfmen73, 02feahw73, nan]"
6,2311.01344v2,"[02ggzyd20, 02tfmen73, 05a1dws80]"
7,2311.01465v2,"[00c9gth79, 05j3snm48, 0409c3r50, 02z9ybv55, 0..."
8,2311.01704v1,"[04jfsqd34, 039cf4q47, 05v727m31]"
9,2311.01892v1,"[05a28rw58, 02tfmen73, 038t36y30]"
10,2311.01922v1,"[035xkbk20, 02tfmen73, 00ntfnx83]"
13,2311.02252v2,"[00kybxq39, 01tmp8f25]"
14,2311.02374v1,"[05a28rw58, 02tfmen73]"


Papers missing target 92 of 139


#### Kings College

First run :

 - 187 / 218 scopus papers contained Imperial College
 -  19 / 218 scopus papers contained King's College

Second run:

 - 232 / 265 scopus papers contained Imperial College
 -  19 / 265 scopus papers contained King's College

In [44]:


def pat_in_text(arx_id, cpat):
    yymm = arx_id.split(".")[0]
    paper_id = arx_id.split("v")[0]
    txt_path = f'txt/arxiv/{yymm}/{arx_id}.txt'
    client = storage.Client(project=PRD_PROJECT)
    bucket = client.bucket(PRD_BUCKET_LOC)
    blob = bucket.blob(txt_path)
    txt_bytes = blob.download_as_bytes()
    file_contents = txt_bytes.decode('utf-8')
    
    if cpat.search(file_contents):
        return True
    return False
    

In [43]:
cpat = re.compile(r"Toulouse") #re.compile(r"[Kk]ing'?s")
found = []
for arx_id in tqdm(check_ids):
    if pat_in_text(arx_id, cpat):
        found.append(arx_id)
len(found)

  0%|          | 0/67 [00:00<?, ?it/s]

65

In [43]:
cpat = re.compile(r"[Kk]ing['’]?s\s+College")
not_found = []
for arx_id in tqdm(check_ids):
    if not pat_in_text(arx_id, cpat):
        not_found.append(arx_id)
print(f"Found pat in {len(found)} of {len(check_ids)} papers.")

not_found

  0%|          | 0/24 [00:00<?, ?it/s]

Found pat in 16 of 24 papers.


['2311.05752v1',
 '2311.10443v2',
 '2311.13339v1',
 '2311.13835v1',
 '2311.14129v1',
 '2311.14177v1',
 '2311.17640v2',
 '2311.17640v3']

In [12]:
text = """
Lorem ipsum dolor sit amet, consectetur
adipiscing elit. Sed do eiusmod tempor
incididunt ut labore et dolore magna
aliqua. Ut enim ad minim veniam, quis
nostrud exercitation ullamco laboris nisi
ut aliquip ex ea commodo consequat. Kings
aute irure dolor in reprehenderit in
King's velit esse cillum dolore eu
fugiat nulla pariatur. Excepteur sint
occaecat cupidatat kings proident, sunt in
culpa qui officia deserunt mollit anim id
est laborum.
"""
pat = re.compile(r"[Kk]ing'?s")
pat.search(text)

<regex.Match object; span=(233, 238), match='Kings'>

In [204]:
all(i not in sub_ids for i in paper_grps['ror'][0])

True

In [112]:
arx_id = '2311.00048v1' #'2311.00018v1'
phase_one.get_single_file_results(arx_id, verbose=True)

[('2311.00048v1',
  'Washington University School of Medicine',
  'St. Louis',
  '04cf69335'),
 ('2311.00048v1', 'Arizona State University', 'Tempe', '03efmqc40')]

In [37]:
target_rors = ['02mp2av58']
res_df[res_df['paper_id'].isin(scopus_ids) & res_df['ror'].isin(target_rors)]

,arx_id,name,ror,paper_id
11587,2311.17969v1,Texas A&M University,02mp2av58,2311.17969
11588,2311.17969v1,New York University,02mp2av58,2311.17969
21437,2311.05877v1,New York University,02mp2av58,2311.05877
22638,2311.08970v1,University of Florida,02mp2av58,2311.08970
24791,2311.03534v2,Microsoft Research,02mp2av58,2311.03534
24792,2311.03534v2,Meta,02mp2av58,2311.03534
36274,2311.18494v1,Yandex LLC,02mp2av58,2311.18494
36276,2311.18494v1,New York University,02mp2av58,2311.18494
44177,2311.16098v1,Meta,02mp2av58,2311.16098
44191,2311.03386v1,New York University,02mp2av58,2311.03386


In [30]:
name_txt = "University of Colorado"
res_df[res_df['paper_id'].isin(scopus_ids) & res_df['name'].str.contains(name_txt)].head()

,arx_id,name,ror,paper_id
1931,2311.01187v1,"University of Colorado Boulder, Boulder",02ttsq026,2311.01187
6729,2311.18020v2,"University of Colorado Boulder, Boulder",02ttsq026,2311.18020
8553,2311.07483v1,"University of Colorado, Boulder",02ttsq026,2311.07483
8755,2311.06424v2,"University of Colorado Boulder,",02ttsq026,2311.06424
14750,2311.13322v2,"University of Colorado Boulder, Boulder",02ttsq026,2311.13322


## rerun small batch

In [72]:
importlib.reload(phase_one)

  0%|          | 0/2091 [00:00<?, ?it/s]

  0%|          | 0/44 [00:00<?, ?it/s]

<module 'phase_one_json' from '/home/jupyter/metadata-vertexai/phase_one_json.py'>

In [73]:
#arx_id

In [74]:
test_id = '2311.00132v1'
phase_one.get_single_file_results(test_id, verbose=True, vverbose=True)

Processing ftp/arxiv/papers/2311/2311.00132.tar.gz
	Processing ftp/arxiv/papers/2311/2311.00132.tar.gz, main.tex
0: \author{E. Bonnetier\textsuperscript{3}}
{\textsuperscript{3} Laboratoire Jean Kuntzmann, Universit\'e de Joseph Fourier, Grenoble, France}
\address
{\textsuperscript{3} Laboratoire Jean Kuntzmann, Universit\'e de Joseph Fourier, Grenoble, France}
\author{M. Courdurier\textsuperscript{2}}
{\textsuperscript{2} Facultad de Matem\'aticas, Pontificia Universidad Cat\'olica de Chile, Santiago, Chile}
\address
{\textsuperscript{2} Facultad de Matem\'aticas, Pontificia Universidad Cat\'olica de Chile, Santiago, Chile}
\author{A. Osses\textsuperscript{1}}
{\textsuperscript{1} DIM-CMM, Universidad de Chile, Santiago, Chile}
\address
{\textsuperscript{1} DIM-CMM, Universidad de Chile, Santiago, Chile}
\author{F. Triki\textsuperscript{3}}
{faouzi.triki@@univ-grenoble-alpes.fr}
 Laboratoire Jean Kuntzmann, Universit
 Facultad de Matem
 DIM-CMM, Universidad de Chile, Santiago, Chile



[('2311.00132v1', 'Université de Joseph Fourier', 'Grenoble', '02aj0kh94'),
 ('2311.00132v1',
  'Pontificia Universidad Católica de Chile',
  'Santiago',
  '04teye511'),
 ('2311.00132v1', 'Universidad de Chile', 'Santiago', '047gc3g35'),
 ('2311.00132v1', 'arXiv', '', '00m2zh467')]

In [ ]:
len("Universit\u00e0")

In [71]:
res_flat_list = stlouis_res_flat
csv_name = 'stlouis.csv'
rerun_df = pd.DataFrame.from_records(res_flat_list, columns=['arx_id', 'name', 'city', 'ror'])
rerun_df.to_csv(csv_name, index=False)
rerun_df.head()
processed_idx = set(rerun_df['arx_id'].unique())

,arx_id,name,city,ror
0,2311.00780v1,University of Maryland,College Park,047s2c258
1,2311.00780v1,Joint Space Science Institute,College Park,01wjew854
2,2311.00780v1,MIT Kavli Institute for Astrophysics and Space...,Cambridge,null
3,2311.00780v1,NASA Goddard Space Flight Center,Greenbelt,0171mag52
4,2311.00780v1,Eureka Scientific,Oakland,04hx2r306


In [138]:
processed_idx = set()
stlouis_res_flat = []

In [139]:
for arx_id in tqdm(check_ids):
    if arx_id in processed_idx:
        continue
    res1 = phase_one.get_single_file_results(arx_id, verbose=False)
    stlouis_res_flat.extend(res1)
    
res_flat_list = stlouis_res_flat
csv_name = 'stlouis2.csv'
rerun_df = pd.DataFrame.from_records(res_flat_list, columns=['arx_id', 'name', 'city', 'ror'])
rerun_df.to_csv(csv_name, index=False)
rerun_df.head()
processed_idx = set(rerun_df['arx_id'].unique())

  0%|          | 0/9 [00:00<?, ?it/s]

,arx_id,name,city,ror
0,2311.03637v1,University of British Columbia,Vancouver,03rmrcq20
1,2311.03637v1,Università degli Studi di Padova,Padova,00240q980
2,2311.03637v1,University College London,Holmbury St Mary,02jx3x895
3,2311.03637v1,INAF Osservatorio Astronomico di Roma,Monte Porzio Catone,02hnp4676
4,2311.03637v1,Massachusetts Institute of Technology,Cambridge,042nb2s44


In [84]:
gemini_res = '''
```json
{"name": "Institute for Astronomy, University of Edinburgh", "city": "Edinburgh", "country": "UK"}
{"name": "Dipartimento di Scienze Matematiche, Fisiche e Informatiche, Universit\u00e0 di Parma", "city": "Parma", "country": "Italy"}
{"name": "Technion Israel Institute of Technology", "city": null, "country": "Israel"}
{"name": "SISSA, International School for Advanced Studies", "city": "Trieste", "country": "Italy"}
{"name": "ICSC - Centro Nazionale di Ricerca in High Performance Computing, Big Data e Quantum Computing", "city": "Bologna", "country": "Italy"}
{"name": "IFPU, Institute for Fundamental Physics of the Universe", "city": "Trieste", "country": "Italy"}
{"name": "INFN Gruppo Collegato di Parma", "city": "Parma", "country": "Italy"}
{"name": "Dipartimento di Fisica \"Aldo Pontremoli\", Universit\u00e0 degli Studi di Milano", "city": "Milano", "country": "Italy"}
{"name": "INAF-IASF Milano", "city": "Milano", "country": "Italy"}
{"name": "School of Physics and Astronomy, Queen Mary University of London", "city": "London", "country": "UK"}
{"name": "Institut de Physique Th\u00e9orique, CEA, CNRS, Universit\u00e9 Paris-Saclay", "city": "Gif-sur-Yvette Cedex", "country": "France"}
{"name": "Institute for Theoretical Particle Physics and Cosmology (TTK), RWTH Aachen University", "city": "Aachen", "country": "Germany"}
{"name": "Department of Physics \"E. Pancini\", University Federico II", "city": "Napoli", "country": "Italy"}
{"name": "Institute of Cosmology and Gravitation, University of Portsmouth", "city": "Portsmouth", "country": "UK"}
{"name": "Dipartimento di Fisica, Universit\u00e0 degli Studi di Torino", "city": "Torino", "country": "Italy"}
{"name": "INFN-Sezione di Torino", "city": "Torino", "country": "Italy"}
{"name": "INAF-Osservatorio Astrofisico di Torino", "city": "Pino Torinese", "country": "Italy"}
{"name": "Higgs Centre for Theoretical Physics, School of Physics and Astronomy, The University of Edinburgh", "city": "Edinburgh", "country": "UK"}
{"name": "Dipartimento di Fisica e Astronomia, Universit\u00e0 di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "INAF-Osservatorio di Astrofisica e Scienza dello Spazio di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "INFN-Sezione di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "Istituto Nazionale di Fisica Nucleare, Sezione di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "Universit\u00e9 de Gen\u00e8ve, D\u00e9partement de Physique Th\u00e9orique and Centre for Astroparticle Physics", "city": "Gen\u00e8ve", "country": "Switzerland"}
{"name": "Universit\u00e9 Paris-Saclay, CNRS, Institut d'astrophysique spatiale", "city": "Orsay", "country": "France"}
{"name": "School of Mathematics and Physics, University of Surrey", "city": "Guildford", "country": "UK"}
{"name": "INAF-Osservatorio Astronomico di Brera", "city": "Milano", "country": "Italy"}
{"name": "Max Planck Institute for Extraterrestrial Physics", "city": "Garching", "country": "Germany"}
{"name": "Dipartimento di Fisica, Universit\u00e0 di Genova", "city": "Genova", "country": "Italy"}
{"name": "INFN-Sezione di Genova", "city": "Genova", "country": "Italy"}
{"name": "INAF-Osservatorio Astronomico di Capodimonte", "city": "Napoli", "country": "Italy"}
{"name": "INFN section of Naples", "city": "Napoli", "country": "Italy"}
{"name": "Instituto de Astrof\u00edsica e Ci\u00eancias do Espa\u00e7o, Universidade do Porto, CAUP", "city": "Porto", "country": "Portugal"}
{"name": "INAF-Osservatorio Astronomico di Roma", "city": "Monteporzio Catone", "country": "Italy"}
{"name": "INFN-Sezione di Roma", "city": "Roma", "country": "Italy"}
{"name": "Institut de F\u00edsica d'Altes Energies (IFAE), The Barcelona Institute of Science and Technology", "city": "Bellaterra", "country": "Spain"}
{"name": "Port d'Informaci\u00f3 Cient\u00edfica", "city": "Bellaterra", "country": "Spain"}
{"name": "Dipartimento di Fisica e Astronomia \"Augusto Righi\" - Alma Mater Studiorum Universit\u00e0 di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "Jodrell Bank Centre for Astrophysics, Department of Physics and Astronomy, University of Manchester", "city": "Manchester", "country": "UK"}
{"name": "European Space Agency/ESRIN", "city": "Frascati", "country": "Italy"}
{"name": "ESAC/ESA", "city": "Villanueva de la Ca\u00f1ada", "country": "Spain"}
{"name": "University of Lyon, Univ Claude Bernard Lyon 1, CNRS/IN2P3, IP2I Lyon", "city": "Villeurbanne", "country": "France"}
{"name": "Aix-Marseille Universit\u00e9, CNRS, CNES, LAM", "city": "Marseille", "country": "France"}
{"name": "Institute of Physics, Laboratory of Astrophysics, Ecole Polytechnique F\u00e9d\u00e9rale de Lausanne (EPFL), Observatoire de Sauverny", "city": "Versoix", "country": "Switzerland"}
{"name": "UCB Lyon 1, CNRS/IN2P3, IUF, IP2I Lyon", "city": "Villeurbanne", "country": "France"}
{"name": "Departamento de F\u00edsica, Faculdade de Ci\u00eancias, Universidade de Lisboa", "city": "Lisboa", "country": "Portugal"}
{"name": "Instituto de Astrof\u00edsica e Ci\u00eancias do Espa\u00e7o, Faculdade de Ci\u00eancias, Universidade de Lisboa", "city": "Lisboa", "country": "Portugal"}
{"name": "Department of Astronomy, University of Geneva", "city": "Versoix", "country": "Switzerland"}
{"name": "INAF-Istituto di Astrofisica e Planetologia Spaziali", "city": "Roma", "country": "Italy"}
{"name": "Department of Physics, Oxford University", "city": "Oxford", "country": "UK"}
{"name": "INFN-Padova", "city": "Padova", "country": "Italy"}
{"name": "Universit\u00e9 Paris-Saclay, Universit\u00e9 Paris Cit\u00e9, CEA, CNRS, AIM", "city": "Gif-sur-Yvette", "country": "France"}
{"name": "Institut d'Estudis Espacials de Catalunya (IEEC)", "city": "Barcelona", "country": "Spain"}
{"name": "Institut de Ciencies de l'Espai (IEEC-CSIC)", "city": "Barcelona", "country": "Spain"}
{"name": "INAF-Osservatorio Astronomico di Trieste", "city": "Trieste", "country": "Italy"}
{"name": "INAF-Osservatorio Astronomico di Padova", "city": "Padova", "country": "Italy"}
{"name": "University Observatory, Faculty of Physics, Ludwig-Maximilians-Universit\u00e4t", "city": "Munich", "country": "Germany"}
{"name": "INFN-Sezione di Milano", "city": "Milano", "country": "Italy"}
{"name": "Institute of Theoretical Astrophysics, University of Oslo", "city": "Oslo", "country": "Norway"}
{"name": "von Hoerner & Sulger GmbH", "city": "Schwetzingen", "country": "Germany"}
{"name": "Technical University of Denmark", "city": "Kgs. Lyngby", "country": "Denmark"}
{"name": "Cosmic Dawn Center (DAWN)", "city": null, "country": "Denmark"}
{"name": "Max-Planck-Institut f\u00fcr Astronomie", "city": "Heidelberg", "country": "Germany"}
{"name": "Department of Physics and Astronomy, University College London", "city": "London", "country": "UK"}
{"name": "Department of Physics and Helsinki Institute of Physics, Gustaf H\u00e4llstr\u00f6min katu 2, 00014 University of Helsinki", "city": "Helsinki", "country": "Finland"}
{"name": "Aix-Marseille Universit\u00e9, CNRS/IN2P3, CPPM", "city": "Marseille", "country": "France"}
{"name": "Jet Propulsion Laboratory, California Institute of Technology", "city": "Pasadena", "country": "USA"}
{"name": "AIM, CEA, CNRS, Universit\u00e9 Paris-Saclay, Universit\u00e9 de Paris", "city": "Gif-sur-Yvette", "country": "France"}
{"name": "Mullard Space Science Laboratory, University College London", "city": "Holmbury St Mary", "country": "UK"}
{"name": "Department of Physics, P.O. Box 64, 00014 University of Helsinki", "city": "Helsinki", "country": "Finland"}
{"name": "Helsinki Institute of Physics, Gustaf H\u00e4llstr\u00f6min katu 2, University of Helsinki", "city": "Helsinki", "country": "Finland"}
{"name": "NOVA optical infrared instrumentation group at ASTRON", "city": "Dwingeloo", "country": "The Netherlands"}
{"name": "Universit\u00e4t Bonn, Argelander-Institut f\u00fcr Astronomie", "city": "Bonn", "country": "Germany"}
{"name": "Dipartimento di Fisica e Astronomia \"Augusto Righi\" - Alma Mater Studiorum Universit\u00e0 di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "Department of Physics, Institute for Computational Cosmology, Durham University", "city": "DH1 3LE", "country": "UK"}
{"name": "European Space Agency/ESTEC", "city": "Noordwijk", "country": "The Netherlands"}
{"name": "Department of Physics and Astronomy, University of Aarhus", "city": "Aarhus C", "country": "Denmark"}
{"name": "Centre for Astrophysics, University of Waterloo", "city": "Waterloo", "country": "Canada"}
{"name": "Department of Physics and Astronomy, University of Waterloo", "city": "Waterloo", "country": "Canada"}
{"name": "Perimeter Institute for Theoretical Physics", "city": "Waterloo", "country": "Canada"}
{"name": "Universit\u00e9 Paris-Saclay, Universit\u00e9 Paris Cit\u00e9, CEA, CNRS, Astrophysique, Instrumentation et Mod\u00e9lisation Paris-Saclay", "city": "Gif-sur-Yvette", "country": "France"}
{"name": "Space Science Data Center, Italian Space Agency", "city": "Roma", "country": "Italy"}
{"name": "Centre National d'Etudes Spatiales -- Centre spatial de Toulouse", "city": "Toulouse Cedex 9", "country": "France"}
{"name": "Institute of Space Science", "city": "M\u0103gurele", "country": "Romania"}
{"name": "Dipartimento di Fisica e Astronomia \"G. Galilei\", Universit\u00e0 di Padova", "city": "Padova", "country": "Italy"}
{"name": "Universit\u00e4ts-Sternwarte M\u00fcnchen, Fakult\u00e4t f\u00fcr Physik, Ludwig-Maximilians-Universit\u00e4t M\u00fcnchen", "city": "M\u00fcnchen", "country": "Germany"}
{"name": "Departamento de F\u00edsica, FCFM, Universidad de Chile", "city": "Santiago", "country": "Chile"}
{"name": "Institute of Space Sciences (ICE, CSIC)", "city": "Barcelona", "country": "Spain"}
{"name": "Satlantis", "city": "Leioa-Bilbao", "country": "Spain"}
{"name": "Centro de Investigaciones Energ\u00e9ticas, Medioambientales y Tecnol\u00f3gicas (CIEMAT)", "city": "Madrid", "country": "Spain"}
{"name": "Instituto de Astrof\u00edsica e Ci\u00eancias do Espa\u00e7o, Faculdade de Ci\u00eancias, Universidade de Lisboa", "city": "Lisboa", "country": "Portugal"}
{"name": "Universidad Polit\u00e9cnica de Cartagena, Departamento de Electr\u00f3nica y Tecnolog\u00eda de Computadoras", "city": "Cartagena", "country": "Spain"}
{"name": "Institut de Recherche en Astrophysique et Plan\u00e9tologie (IRAP), Universit\u00e9 de Toulouse, CNRS, UPS, CNES", "city": "Toulouse", "country": "France"}
{"name": "Kapteyn Astronomical Institute, University of Groningen", "city": "Groningen", "country": "The Netherlands"}
{"name": "INFN-Bologna", "city": "Bologna", "country": "Italy"}
{"name": "Infrared Processing and Analysis Center, California Institute of Technology", "city": "Pasadena", "country": "USA"}
{"name": "INAF, Istituto di Radioastronomia", "city": "Bologna", "country": "Italy"}
{"name": "Instituto de Astrof\u00edsica de Canarias", "city": "San Crist\u00f3bal de La Laguna", "country": "Spain"}
{"name": "Institut f\u00fcr Theoretische Physik, University of Heidelberg", "city": "Heidelberg", "country": "Germany"}
{"name": "Universit\u00e9 St Joseph; Faculty of Sciences", "city": "Beirut", "country": "Lebanon"}
{"name": "Institut d'Astrophysique de Paris", "city": "Paris", "country": "France"}
{"name": "Junia, EPA department", "city": "Lille", "country": "France"}
{"name": "INFN, Sezione di Trieste", "city": "Trieste TS", "country": "Italy"}
{"name": "Instituto de F\u00edsica Te\u00f3rica UAM-CSIC", "city": "Madrid", "country": "Spain"}
{"name": "CERCA/ISO, Department of Physics, Case Western Reserve University", "city": "Cleveland", "country": "USA"}
{"name": "Laboratoire Univers et Th\u00e9orie, Observatoire de Paris, Universit\u00e9 PSL, Universit\u00e9 Paris Cit\u00e9, CNRS", "city": "Meudon", "country": "France"}
{"name": "Dipartimento di Fisica e Scienze della Terra, Universit\u00e0 degli Studi di Ferrara", "city": "Ferrara", "country": "Italy"}
{"name": "Istituto Nazionale di Fisica Nucleare, Sezione di Ferrara", "city": "Ferrara", "country": "Italy"}
{"name": "Institut d'Astrophysique de Paris, UMR 7095, CNRS, and Sorbonne Universit\u00e9", "city": "Paris", "country": "France"}
{"name": "Dipartimento di Fisica - Sezione di Astronomia, Universit\u00e0 di Trieste", "city": "Trieste", "country": "Italy"}
{"name": "Minnesota Institute for Astrophysics, University of Minnesota", "city": "Minneapolis", "country": "USA"}
{"name": "Universit\u00e9 C\u00f4te d'Azur, Observatoire de la C\u00f4te d'Azur, CNRS, Laboratoire Lagrange", "city": "Nice cedex 4", "country": "France"}
{"name": "Institute Lorentz, Leiden University", "city": "Leiden", "country": "The Netherlands"}
{"name": "Institute for Astronomy, University of Hawaii", "city": "Honolulu", "country": "USA"}
{"name": "Department of Physics & Astronomy, University of California Irvine", "city": "Irvine", "country": "USA"}
{"name": "Department of Astronomy & Physics and Institute for Computational Astrophysics, Saint Mary's University", "city": "Halifax", "country": "Canada"}
{"name": "Departamento F\u00edsica Aplicada, Universidad Polit\u00e9cnica de Cartagena", "city": "Cartagena", "country": "Spain"}
{"name": "Universit\u00e9 Paris Cit\u00e9, CNRS, Astroparticule et Cosmologie", "city": "Paris", "country": "France"}
{"name": "Department of Computer Science, Aalto University", "city": "Espoo", "country": "Finland"}
{"name": "Department of Physics and Astronomy, Vesilinnantie 5, 20014 University of Turku", "city": "Turku", "country": "Finland"}
{"name": "Serco for European Space Agency (ESA)", "city": "Villanueva de la Ca\u00f1ada", "country": "Spain"}
{"name": "ARC Centre of Excellence for Dark Matter Particle Physics", "city": "Melbourne", "country": "Australia"}
{"name": "Centre for Astrophysics & Supercomputing, Swinburne University of Technology", "city": "Victoria", "country": "Australia"}
{"name": "W.M. Keck Observatory", "city": "Kamuela", "country": "USA"}
{"name": "Department of Physics and Astronomy, University of the Western Cape", "city": "Bellville", "country": "South Africa"}
{"name": "Oskar Klein Centre for Cosmoparticle Physics, Department of Physics, Stockholm University", "city": "Stockholm", "country": "Sweden"}
{"name": "Astrophysics Group, Blackett Laboratory, Imperial College London", "city": "London", "country": "UK"}
{"name": "Univ. Grenoble Alpes, CNRS, Grenoble INP, LPSC-IN2P3", "city": "Grenoble", "country": "France"}
{"name": "Dipartimento di Fisica, Sapienza Universit\u00e0 di Roma", "city": "Roma", "country": "Italy"}
{"name": "Centro de Astrof\u00edsica da Universidade do Porto", "city": "Porto", "country": "Portugal"}
{"name": "Zentrum f\u00fcr Astronomie, Universit\u00e4t Heidelberg", "city": "Heidelberg", "country": "Germany"}
{"name": "Dipartimento di Fisica, Universit\u00e0 di Roma Tor Vergata", "city": "Roma", "country": "Italy"}
{"name": "INFN, Sezione di Roma 2", "city": "Roma", "country": "Italy"}
{"name": "Institute of Astronomy, University of Cambridge", "city": "Cambridge", "country": "UK"}
{"name": "Institute for Computational Science, University of Zurich", "city": "Zurich", "country": "Switzerland"}
{"name": "Department of Astrophysical Sciences, Peyton Hall, Princeton University", "city": "Princeton", "country": "USA"}
{"name": "Niels Bohr Institute, University of Copenhagen", "city": "Copenhagen", "country": "Denmark"}
```
'''
found_institutions = []
verbose=True
for row in gemini_res.strip().splitlines():
    if row:
        if row.startswith('`'):
            continue
        try:
            found_institutions.append(json.loads(r'{}'.format(row)))
        except json.JSONDecodeError as e:
            if verbose:
                print(f"JSONDecodeError: {e} on {arx_id} at {row}")
            pass

JSONDecodeError: Expecting ',' delimiter: line 1 column 35 (char 34) on 2311.13529v2 at {"name": "Dipartimento di Fisica "Aldo Pontremoli", Università degli Studi di Milano", "city": "Milano", "country": "Italy"}
JSONDecodeError: Expecting ',' delimiter: line 1 column 34 (char 33) on 2311.13529v2 at {"name": "Department of Physics "E. Pancini", University Federico II", "city": "Napoli", "country": "Italy"}
JSONDecodeError: Expecting ',' delimiter: line 1 column 48 (char 47) on 2311.13529v2 at {"name": "Dipartimento di Fisica e Astronomia "Augusto Righi" - Alma Mater Studiorum Università di Bologna", "city": "Bologna", "country": "Italy"}
JSONDecodeError: Expecting ',' delimiter: line 1 column 48 (char 47) on 2311.13529v2 at {"name": "Dipartimento di Fisica e Astronomia "Augusto Righi" - Alma Mater Studiorum Università di Bologna", "city": "Bologna", "country": "Italy"}
JSONDecodeError: Expecting ',' delimiter: line 1 column 48 (char 47) on 2311.13529v2 at {"name": "Dipartimento di Fisi

In [123]:
gemini_res = '''
{"name": "Universit\\`a di Padova", "city": "Padova", "country": "Italy"}
{"name": "Department of Physics \\"E. Pancini\\", University Federico II", "city": "Napoli", "country": "Italy"}
'''
institutions_found = []
verbose=True
for row in gemini_res.strip().splitlines():
    if row:
        if row.startswith('`'):
            continue
        try:
            institutions_found.append(json.loads(row))
        except json.JSONDecodeError as e:
            try:
                institutions_found.append(json.loads(r"{}".format(row).replace('\\', '\\\\')))
            except json.JSONDecodeError:
                if verbose:
                    print(f"JSONDecodeError: {e} on {arx_id} at {row}")
                pass
institutions_found

[{'name': 'Universit\\`a di Padova', 'city': 'Padova', 'country': 'Italy'},
 {'name': 'Department of Physics "E. Pancini", University Federico II',
  'city': 'Napoli',
  'country': 'Italy'}]

## Scratch

In [ ]:
ror
(ror in skip_inst)
ror_df[ror_df['ror']==ror]
ror_map_df[ror_map_df['ror']==ror]

In [ ]:
ror_df.loc[ror_df['ror']==ror,'name'].iloc[0]

In [ ]:
res = {
    "ror": ror,
    "TP": len(res_ids.intersection(scopus_ids)), 
    "FP": len(res_ids - scopus_ids), 
    "FN": len(scopus_ids - res_ids), 
    "TN": len((scopus_all - scopus_ids) - res_ids)
}

In [ ]:
list(itr.islice(scopus_ids, 10))

In [ ]:
list(itr.islice(res_ids, 10))

In [ ]:
#%%time
#res_dict = {}
#for arx_id in tqdm(false_positive): #scopus_positive:
#    #print(arx_id)
#    paper_id = arx_id.split("v")[0]
#    res = phase_one.send_one_submission_to_gemini(arx_id)
#    #print(f"\n{paper_id}\n{res}")
#    res_dict[paper_id] = res
#

### Phase 2 Name --> ROR id

Based on FAISS

In [22]:
importlib.reload(phase_one)

<module 'phase_one_json' from '/home/jupyter/metadata-vertexai/phase_one_json.py'>

In [46]:
ror_finder = phase_one.ROR_FINDER

In [97]:
ror_finder20 = phase_one.rorFinder(doc_k=20)

In [47]:
ror_finder.qa_chain.invoke({"query": 'Washington University School of Medicine, St. Louis'})

{'query': 'Washington University School of Medicine, St. Louis',
 'result': '01yc7t268\n',
 'source_documents': [Document(id='d7814266-6091-4411-bf86-cc17c0571119', metadata={}, page_content='Washington University in St. Louis School of Medicine — https://ror.org/01yc7t268'),
  Document(id='e4aefc05-a496-4dec-bff2-62395d896a8f', metadata={}, page_content='Washington University in St. Louis School of Medicine, St Louis — https://ror.org/01yc7t268'),
  Document(id='7bfd1030-b3e3-494a-99cc-0be8d983cd37', metadata={}, page_content='University of Washington School of Medicine — https://ror.org/00cvxb145'),
  Document(id='da94fa35-a5d4-4840-bd14-6b69ada4cc5a', metadata={}, page_content='Washington University Medical Center, St Louis — https://ror.org/036c27j91'),
  Document(id='3a73cd45-6cd6-4ba6-b76a-bbc9db2c2edd', metadata={}, page_content='University of Washington School of Medicine, Seattle — https://ror.org/00cvxb145')]}

In [99]:
ror_finder20.qa_chain.invoke({"query": 'Université Paris-Saclay, Gif-sur-Yvette'})

{'query': 'Université Paris-Saclay, Gif-sur-Yvette',
 'result': '03xjwb503\n',
 'source_documents': [Document(id='5c27f3ec-12dc-4abf-a961-74c855847901', metadata={}, page_content='Université Paris-Saclay, Gif-sur-Yvette — https://ror.org/03xjwb503'),
  Document(id='1a5b13ff-31ad-44e3-84c9-35f73954e741', metadata={}, page_content='University of Paris-Saclay, Gif-sur-Yvette — https://ror.org/03xjwb503'),
  Document(id='9a6e6d3e-500e-4769-b22f-76290d47c594', metadata={}, page_content='Universitat París-Saclay, Gif-sur-Yvette — https://ror.org/03xjwb503'),
  Document(id='a6063f78-43e9-4f27-a3f6-b1e619fa4b18', metadata={}, page_content='École Normale Supérieure Paris-Saclay, Gif-sur-Yvette — https://ror.org/00hx6zz33'),
  Document(id='fa7eb3f4-2c67-4469-839f-d6645d05a02f', metadata={}, page_content='CEA Paris-Saclay, Gif-sur-Yvette — https://ror.org/03n15ch10'),
  Document(id='4b13cad6-2777-4db9-89e6-b8e70111983f', metadata={}, page_content="Institut de Recherche sur les Lois Fondamentales 

In [209]:
ror_finder5.qa_chain.invoke({"query": 'New York University, New York'})

{'query': 'New York University, New York',
 'result': '0190ak572\n',
 'source_documents': [Document(id='6e603100-4e7c-447c-8e6c-d9f01ab5aa5f', metadata={}, page_content='York College, City University of New York, New York — https://ror.org/015a1ak54'),
  Document(id='a938e6e9-78ca-4711-9b10-485ed79801ff', metadata={}, page_content='City University of New York, New York — https://ror.org/00453a208'),
  Document(id='5cdb0189-0b74-46c8-89e7-f40fb3e3b4d0', metadata={}, page_content='New York University, New York — https://ror.org/0190ak572'),
  Document(id='5368175c-fbb5-4dfc-957b-e8840930398c', metadata={}, page_content='York University, York — https://ror.org/022jz8688'),
  Document(id='d07f8630-c036-4e23-8095-4a3fc3ca86d8', metadata={}, page_content='University of York, York — https://ror.org/04m01e293')]}

In [212]:
%%time
ror_finder5.get_ror('New York University, New York')

CPU times: user 8 µs, sys: 1e+03 ns, total: 9 µs
Wall time: 13.4 µs


'0190ak572'

In [ ]:
%%time
ror_finder.get_ror('Universitas Indonesia,')

## Prompt Experiments

In [67]:
input_text = '''
 \author{
    Sonish Sivarajkumar, MS\textsuperscript{1}\textsuperscript{,2}\thanks{Present address: School of Computing and Information, University of Pittsburgh, Pennsylvania, PA, USA. Work was done while at Molecular Robotics, Kerala, India.},
    Pratyush Tandale, MS\textsuperscript{3}, 
    Ankit Bhardwaj, BS\textsuperscript{4}, \\
    Kipp W. Johnson, MD,PhD\textsuperscript{5}, 
    Anoop Titus, MD\textsuperscript{6}, 
    Benjamin S. Glicksberg, PhD\textsuperscript{7},\\
    Shameer Khader, PhD, MPH\textsuperscript{8}\textsuperscript{\dag}, 
    Kamlesh K. Yadav, PhD\textsuperscript{9, 10}\textsuperscript{\dag}, \\
    Lakshminarayanan Subramanian, PhD\textsuperscript{4}\thanks{Corresponding authors: shameer.khader20@imperial.ac.uk, kamlesh.yadav@tamu.edu, lakshmi@cs.nyu.edu}
}
,
    Pratyush Tandale, MS
,
    Pratyush Tandale, MS
, 
    Ankit Bhardwaj, BS
, 
, 
    Anoop Titus, MD
, 
    Benjamin S. Glicksberg, PhD
,
, 
    Kamlesh K. Yadav, PhD
, 
    Kamlesh K. Yadav, PhD
, 
, 


Corresponding authors
Molecular Robotics, Kerala, India; 
School of Computing and Information, University of Pittsburgh, Pennsylvania, PA, USA; 
Health Informatics  &  Data Science, Georgetown University, Washington DC, USA; 
Department of Computer Science, Courant Institute of Mathematical Sciences, New York University, New York, NY, USA; 
Institute for Next Generation Healthcare, Mount Sinai Health System, New York, NY, USA; 
Department of Preventive Cardiology, DeBakey Heart  &  Vascular Center, Houston Methodist, Houston, TX, USA; 
Hasso Plattner Institute for Digital Health, Icahn School of Medicine at Mount Sinai, New York, NY, USA; 
Faculty of Medicine, Imperial College London, London, UK; 
School of Engineering Medicine,  Texas A & M University, Houston, TX, USA; 
Department of Translational Medical Sciences, Center for Genomic and Precision Medicine, Texas A & M University, Houston, TX, USA; 
'''.strip()

input_text = r'''
\author{Pascal Auscher}
\address
{Universit{\'e} Paris-Saclay, CNRS, Laboratoire de Math\'{e}matiques d'Orsay, 91405 Orsay, France}
\author{Hedong Hou}
\address
{Universit{\'e} Paris-Saclay, CNRS, Laboratoire de Math\'{e}matiques d'Orsay, 91405 Orsay, France}
'''.strip()

input_text3 ='''
Universit\'e Paris-Saclay, Universit\'e Paris Cit\'e, CEA, CNRS, Astrophysique, Instrumentation et Mod\'elisation Paris-Saclay, 91191 Gif-sur-Yvette, France\label{aff80}
\and
Space Science Data Center, Italian Space Agency, via del Politecnico snc, 00133 Roma, Italy\label{aff81}
\and
Centre National d'Etudes Spatiales -- Centre spatial de Toulouse, 18 avenue Edouard Belin, 31401 Toulouse Cedex 9, France\label{aff82}
\and
Dipartimento di Fisica e Astronomia "G. Galilei", Universit\`a di Padova, Via Marzolo 8, 35131 Padova, Italy\label{aff84}
\and
Universit\"ats-Sternwarte M\"unchen, Fakult\"at f\"ur Physik, Ludwig-Maximilians-Universit\"at M\"unchen, Scheinerstrasse 1, 81679 M\"unchen, Germany\label{aff85}
\and
Departamento de F\'isica, FCFM, Universidad de Chile, Blanco Encalada 2008, Santiago, Chile\label{aff86}
\and
Institute of Space Sciences (ICE, CSIC), Campus UAB, Carrer de Can Magrans, s/n, 08193 Barcelona, Spain\label{aff87}
\and
Satlantis, University Science Park, Sede Bld 48940, Leioa-Bilbao, Spain\label{aff88}
\and
Centro de Investigaciones Energ\'eticas, Medioambientales y Tecnol\'ogicas (CIEMAT), Avenida Complutense 40, 28040 Madrid, Spain\label{aff89}
\and
Instituto de Astrof\'isica e Ci\^encias do Espa\c{c}o, Faculdade de Ci\^encias, Universidade de Lisboa, Tapada da Ajuda, 1349-018 Lisboa, Portugal\label{aff90}
'''

inst_list = '''
University of Pittsburgh, Pennsylvania
Georgetown University, Washington DC
New York University, New York
Mount Sinai Health System, New York
Houston Methodist, Houston
Icahn School of Medicine at Mount Sinai, New York
Imperial College London, London
Texas A & M University, Houston
'''.strip()



VERIFY_TEMPLATE = """
Match the institution names in the LIST_OF_NAMES with the contents of the SOURCE_TEXT.
Answer "True" if ALL the institutions in the LIST_OF_NAMES are present in the SOURCE_TEXT, otherwise answer "False"\n
Only respond with "True" or "False".
### LIST_OF_NAMES:\n
{inst_list}\n\n

### SOURCE_TEXT:\n
{source_text}\n\n
""".strip()

VERIFY_TEMPLATE = """
Find all the organizations and any associated locations mentioned in the SOURCE_TEXT.
When organizations are listed together at an address, assume the location information ONLY applies to the immediately preceeding organization.

2. Output Format:
    - Each organization and location, if any, should be on a separate line with NO extra numbering, punctuation, or bullet points.
    - Do NOT include explanations, descriptions, or any other text — ONLY the organization list or 'null'
    - The output should be plain text with no markdown
    - Deduplicate the organization list 
    - Follow this pseudocode to generate the output:
    ```
    if no organization are found, then output "null".
    else
        for each organization
            if the organization is associated with a location
                if the location includes a city and country:
                    output a json object with this format: {{"name":organization, "city":city , "country":country}}
                else:
                   output a json object with this format: {{"name':organization, "city":city}}
            else 
                output a json object with this format: {{"name":organization}}
    ```

### SOURCE_TEXT:\n
{source_text}\n\n
""".strip()

foo = '''
___STRICT OUTPUT REQUIREMENTS:
Output Format:
    - Each organization should be on a separate line with NO extra numbering, punctuation, or bullet points.
    - Do NOT include explanations, descriptions, or any other text — ONLY the organization list or 'null'
    - The output should be plain text with no markdown
    - Deduplicate the organization list 
    - Follow this pseudocode to generate the output:

    if no organization are found, then output "null".
    else
        for each organization
            
            if organization is associated with a city
                if the city includes a country:
                    output a json object with this format: {{"name":organization, "city":city , "country":country}}
                else:
                   output a json object with this format: {{"name':organization, "city":city}}
            else 
                output a json object with this format: {{"name":organization}}
'''


VERIFY_TEMPLATE2 = """
You are an expert in recognizing organization names and associated locations in text and latex input.

Identify the authors' organizations in the following INPUT TEXT below.
When organizations are listed together at an address, assume the location information ONLY applies to the immediately preceeding organization.

___INPUT_TEXT:\n
{source_text}\n\n
""".strip()

foo = """
### OUTPUT FORMAT:
 - Convert any LaTeX character macros to utf-8.
 - print the number of organizations found
 - List each organization and its location as a separate item.
 - If the organization is not associated with a location in the SOURCE_TEXT, do not include location information for that organization

 - print the number of organizations found
 - explain your reasoning

"""

VERIFY_TEMPLATE = """
Output a list of organizations from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.
Follow the STEPS below:

### REQUIREMENTS
 - Consider each organization separately.
 - Convert any LaTeX character macros to utf-8.

### STEP 1:
 - Find all potential organizations AND, if present, any associated locations in the SOURCE_TEXT.
 - When organizations are listed together at an address, treat each organization as a separate entity.
 - When organizations are listed together at an address, assume the location information ONLY applies to the immediately preceeding organization.

### OUTPUT_FORMAT:
 - print the number of organizations found
 - List each organization and its location as a separate item.
 - 
 
### SOURCE_TEXT:\n
{source_text}\n\n
""".strip()





VERIFY_TEMPLATE = """
TASK: Follow the directions to generate output from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.
Follow the directions below:

 - Find all potential organizations AND, if present, any associated locations in the SOURCE_TEXT.
 - When organizations are listed together at an address, treat each organization as a separate entity.
 - When organizations are listed together at an address, assume the location information ONLY applies to the last organization in the list.

### OUTPUT_FORMAT:
 - render any LaTeX in organization names to unicode
 - The output should be plain text
 - Do not use markdown
 - Follow this pseudocode to generate the output:
```
    if no organizations are found, then output "null".
    else
        for each organization
            let org_name = name of the organization
            if organization is associated with a city
                if the city includes a country:
                    output a string with this format: {{"name":org_name, "city":city , "country":country}}
                else:
                   output a string with this format: {{"name':org_name, "city":city}}
            else 
                output a string with this format: {{"name":org_name}}
```

### SOURCE_TEXT:\n
{source_text}\n\n
""".strip()

input_text3 = '''
\author{Ramon Cardias}
\affiliation
{Department of Applied Physics, School of Engineering Sciences, KTH Royal Institute of Technology, AlbaNova University Center, SE-10691 Stockholm, Sweden}
\affiliation
{Instituto de Física, Universidade Federal Fluminense, 24210-346, Niterói RJ, Brazil}
\author{Simon Streib}
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Zhiwei Lu}
\affiliation
{Department of Applied Physics, School of Engineering Sciences, KTH Royal Institute of Technology, AlbaNova University Center, SE-10691 Stockholm, Sweden}
\author{Manuel Pereiro }
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Anders Bergman}
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Erik Sj\"oqvist }
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Cyrille Barreteau}
\affiliation
{Universit\'e Paris-Saclay, CEA, CNRS, SPEC, 91191, Gif-sur-Yvette, France}
\author{Anna Delin }
\affiliation
{Department of Applied Physics, School of Engineering Sciences, KTH Royal Institute of Technology, AlbaNova University Center, SE-10691 Stockholm, Sweden}
\affiliation
{ Swedish e-Science Research Center (SeRC), KTH Royal Institute of Technology, SE-10044 Stockholm, Sweden}
\affiliation
{Wallenberg Initiative Materials Science for Sustainability (WISE), KTH Royal Institute of Technology, SE-10044 Stockholm, Sweden}
\author{Olle Eriksson }
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\affiliation
{Wallenberg Initiative Materials Science for Sustainability (WISE), Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Danny Thonig}
\affiliation
{School of Science and Technology, \"Orebro University, SE-70182 Örebro,
Sweden}
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
'''
input_text3 = '''
\title[Large-scale Magnetorotational Dynamo]{Magnetorotational dynamo can generate large-scale vertical magnetic fields in 3D GRMHD simulations of accreting black holes}



\author[J. Jacquemin-Ide et al.]{
Jonatan Jacquemin-Ide,$^{1}$\thanks{E-mail: jonatan.jacqueminide@northwestern.edu}
François Rincon, $^{2}$
Alexander Tchekhovskoy,$^{1}$
and Matthew Liska$^{3,4}$
\\

$^{1}$Center for Interdisciplinary Exploration $\&$ Research in Astrophysics (CIERA), Physics and Astronomy, Northwestern University, Evanston, IL 60202, USA\\
$^{2}$Institut de Recherche en Astrophysique et Planétologie (IRAP), Université de Toulouse, CNRS, UPS, Toulouse, France\\
$^{3}$Institute for Theory and Computation, Harvard University, 60 Garden Street, Cambridge, MA 02138, USA\\
$^{4}$Center for Relativistic Astrophysics, Georgia Institute of Technology, Howey Physics Bldg, 837 State St NW, Atlanta, GA 30332, USA\\
}


\date{Accepted XXX. Received YYY; in original form ZZZ}


\pubyear{2015}


\begin{document}
'''

input_text4 = '''
\author[J. Jacquemin-Ide et al.]{
Jonatan Jacquemin-Ide,$^{1}$\thanks{E-mail: jonatan.jacqueminide@northwestern.edu}
François Rincon, $^{2}$
Alexander Tchekhovskoy,$^{1}$
and Matthew Liska$^{3,4}$
\\
$^{1}$Center for Interdisciplinary Exploration $\&$ Research in Astrophysics (CIERA), Physics and Astronomy, Northwestern University, Evanston, IL 60202, USA\\
$^{2}$Institut de Recherche en Astrophysique et Planétologie (IRAP), Université de Toulouse, CNRS, UPS, Toulouse, France\\
$^{3}$Institute for Theory and Computation, Harvard University, 60 Garden Street, Cambridge, MA 02138, USA\\
$^{4}$Center for Relativistic Astrophysics, Georgia Institute of Technology, Howey Physics Bldg, 837 State St NW, Atlanta, GA 30332, USA\\
}


\date{Accepted XXX. Received YYY; in original form ZZZ}


\pubyear{2015}


\begin{document}
'''

VERIFY_TEMPLATE = """
TASK: Follow the directions to generate output from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.
Follow the directions below:
 - Find all potential organizations in the SOURCE_TEXT.
 - Expand abbreviations and acronyms of potential organization names.
 - When organizations are listed together at an address, include each organization as a separate item.
 - When organizations are listed together at an address, include any acronyms as a separate entity.
 - Identify any locations associated explicity associated with any of the potential organizations.
 - When organizations are listed together at a single address, ONLY associate the address with the last organization in the list.

### OUTPUT_FORMAT:
 - The output should be valid utf-8 line json
 - Output one json Object per line.
 - Do not return a json Array.
 - Replace any latex escape sequences in the output with utf-8 characters
 - double-escape all backslashes
 - Only report the main organizations like universities, universi, commissions, foundations or corporations.
 - Ignore sub-units like department, dipartimento, or college.
 - Follow this pseudocode to generate the output:
```
    if no organizations are found, then output "null".
    else
        for each organization
            let org_name be the organization name.
            let city be "" unless you identied a city for this organization
            let country be "" unless you identified a country location for this organization
            output a json Object with this format: {{"name":org_name, "city":city , "country":country}}
```
### SOURCE_TEXT:
{source_text}\n\n
""".strip()


res = phase_one.verify_with_gemini_api(inst_list, input_text4, template=VERIFY_TEMPLATE)
print(res)
for row in res.strip().splitlines():
    if row:
        try:
            j = json.loads(row)
            if j is None:
                print("is None")
            else:
                print(json.loads(row))
        except json.JSONDecodeError as e:
            try:
                print(json.loads(r"{}".format(row)))
            except:
                print(f"Error: {repr(row)}")
            pass

```json
{"name": "Northwestern University", "city": "Evanston", "country": "USA"}
{"name": "Université de Toulouse", "city": "Toulouse", "country": "France"}
{"name": "Harvard University", "city": "Cambridge", "country": "USA"}
{"name": "Georgia Institute of Technology", "city": "Atlanta", "country": "USA"}
```

Error: '```json'
{'name': 'Northwestern University', 'city': 'Evanston', 'country': 'USA'}
{'name': 'Université de Toulouse', 'city': 'Toulouse', 'country': 'France'}
{'name': 'Harvard University', 'city': 'Cambridge', 'country': 'USA'}
{'name': 'Georgia Institute of Technology', 'city': 'Atlanta', 'country': 'USA'}
Error: '```'


In [62]:
input_text3 = '''
$^{1}$Center for Interdisciplinary Exploration $\&$ Research in Astrophysics (CIERA), Physics and Astronomy, Northwestern University, Evanston, IL 60202, USA\\
$^{2}$Institut de Recherche en Astrophysique et Planétologie (IRAP), Université de Toulouse, CNRS, UPS, Toulouse, France\\
$^{3}$Institute for Theory and Computation, Harvard University, 60 Garden Street, Cambridge, MA 02138, USA\\
$^{4}$Center for Relativistic Astrophysics, Georgia Institute of Technology, Howey Physics Bldg, 837 State St NW, Atlanta, GA 30332, USA\\
'''
VERIFY_TEMPLATE = """
TASK: Follow the directions to generate output from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.
Follow the directions below:
 - Find all potential organizations in the SOURCE_TEXT.
 - Expand abbreviations and acronyms of potential organization names.
 - When organizations are listed together at an address, treat each organization as a separate entity.
 - When organizations are listed together at an address, expand any acronyms as a separate entity.
 - Identify any locations associated explicity associated with any of the potential organizations.
 - When organizations are listed together at a single address, ONLY associate the address with the last organization in the list.
 - Ignore any sub-units like departments or colleges.

### OUTPUT_FORMAT:
 - The output should be valid utf-8 line json
 - Output one json Object per line.
 - Do not return a json Array.
 - Replace any latex escape sequences in the output with utf-8 characters
 - double-escape all backslashes
 - Only report the main organizations like universities, universi, commissions, foundations or corporations.
 - Ignore sub-units like department, dipartimento, or college.
 - Follow this pseudocode to generate the output:
```
    if no organizations are found, then output "null".
    else
        for each organization
            let org_name be the organization name.
            let city be "" unless you identied a city for this organization
            let country be "" unless you identified a country location for this organization
            output a json Object with this format: {{"name":org_name, "city":city , "country":country}}

### SOURCE_TEXT:
{source_text}\n\n
""".strip()

VERIFY_TEMPLATE2 = '''
identify the organizations mentioned in the SOURCE_TEXT.
list the full name of the organizations

### SOURCE_TEXT:
{source_text}\n\n
'''

res = phase_one.verify_with_gemini_api(inst_list, input_text3, template=VERIFY_TEMPLATE)
print(res)
for row in res.strip().splitlines():
    if row:
        try:
            j = json.loads(row)
            if j is None:
                print("is None")
            else:
                print(json.loads(row))
        except json.JSONDecodeError as e:
            try:
                print(json.loads(r"{}".format(row)))
            except:
                print(f"Error: {repr(row)}")
            pass

```json
{"name": "Center for Interdisciplinary Exploration and Research in Astrophysics", "city": "Evanston", "country": "USA"}
{"name": "Northwestern University", "city": "Evanston", "country": "USA"}
{"name": "Institut de Recherche en Astrophysique et Planétologie", "city": "Toulouse", "country": "France"}
{"name": "Université de Toulouse", "city": "Toulouse", "country": "France"}
{"name": "CNRS", "city": "Toulouse", "country": "France"}
{"name": "UPS", "city": "Toulouse", "country": "France"}
{"name": "Institute for Theory and Computation", "city": "Cambridge", "country": "USA"}
{"name": "Harvard University", "city": "Cambridge", "country": "USA"}
{"name": "Center for Relativistic Astrophysics", "city": "Atlanta", "country": "USA"}
{"name": "Georgia Institute of Technology", "city": "Atlanta", "country": "USA"}
```

Error: '```json'
{'name': 'Center for Interdisciplinary Exploration and Research in Astrophysics', 'city': 'Evanston', 'country': 'USA'}
{'name': 'Northwestern Universit

In [18]:
input_text3 = '''
$$ R-2Ric(X,X)+2\frac{\mathrm{det}(\mathrm{Hess} f)}{|\nabla
f|^2}\le R-2Ric(X,X)+\frac{(1-R+Ric(X,X))^2}{2|\nabla f|^2}<1$$'''
VERIFY_TEMPLATE = """
How would the LaTeX equations in the SOURCE_TEXT spoken in English?
 
### SOURCE_TEXT:
{source_text}\n\n
""".strip()

inst_list = None

res = phase_one.verify_with_gemini_api(inst_list, input_text3, template=VERIFY_TEMPLATE)
print(res)

The LaTeX equation would be spoken as follows:

"R minus two times the Ricci curvature of X and X, plus two times the determinant of the Hessian of f, divided by the magnitude of the gradient of f squared, is less than or equal to R minus two times the Ricci curvature of X and X, plus (one minus R plus the Ricci curvature of X and X) squared, divided by two times the magnitude of the gradient of f squared, which is less than one."



In [84]:
arx_id = '2311.04844v2'
old_template = phase_one.PROMPT_TEMPLATE
phase_one.PROMPT_TEMPLATE = """
TASK: Follow the directions to generate output from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.
Follow the directions below:
 - Find all potential organizations in the SOURCE_TEXT.
 - When organizations are listed together at an address, treat each organization as a separate entity.
 - Identify any locations associated explicity associated with any of the potential organizations.
 - When organizations are listed together at a single address, ONLY associate the address with the last organization in the list.

### OUTPUT_FORMAT:
 - The output should be valid utf-8 line json
 - replace any latex characters in the output with utf-8 equivalent
 - Do not use markdown
 - Do not enclose output in backticks
 - Follow this pseudocode to generate the output:
```
    if no organizations are found, then output "null".
    else
        for each organization
            let org_name be the organization name
            let city be "" unless you identied a city for this organization
            let country be "" unless you identified a country location for this organization
            output a string with this format: {{"name":org_name, "city":city , "country":country}}

### SOURCE_TEXT:\n
{input_text}\n\n
""".strip()
res = phase_one.get_single_file_results(arx_id, verbose=True)
phase_one.PROMPT_TEMPLATE = old_template

res
#print(json.loads(res[0][1]))


Processing ftp/arxiv/papers/2311/2311.04844.tar.gz
0: \author{Pascal Auscher}
\address
{Universit{\'e} Paris-Saclay, CNRS, Laboratoire de Math\'{e}matiques d'Orsay, 91405 Orsay, France}
\author{Hedong Hou}
\address
{Universit{\'e} Paris-Saclay, CNRS, Laboratoire de Math\'{e}matiques d'Orsay, 91405 Orsay, France}

{"name": "Universit\u00e9 Paris-Saclay", "city": "", "country": ""}
{"name": "CNRS", "city": "", "country": ""}
{"name": "Laboratoire de Math\u00e9matiques d'Orsay", "city": "Orsay", "country": "France"}



[('2311.04844v2', 'Université Paris-Saclay', '', 'null'),
 ('2311.04844v2', 'CNRS', '', '02feahw73'),
 ('2311.04844v2',
  "Laboratoire de Mathématiques d'Orsay",
  'Orsay',
  '03ab0zs98')]

In [85]:
print(phase_one.PROMPT_TEMPLATE_V4_edited)

AttributeError: module 'phase_one_json' has no attribute 'PROMPT_TEMPLATE_V4_edited'

In [23]:
phase_one.get_single_file_results('2311.03508v2', verbose=True, vverbose=True)

Processing ftp/arxiv/papers/2311/2311.03508.tar.gz
	Processing ftp/arxiv/papers/2311/2311.03508.tar.gz, multiscale_neuro_glia_neural_network_learning.tex


KeyboardInterrupt: 

### Clean up results

In [ ]:
res_df = pd.read_csv("gs://institutional-extract-scratch/output/2311_db_all.csv.zip")

In [ ]:
res_df.shape
res_df.head()

In [ ]:
# probably html formatted files
error_files = res_df[res_df['name']=='error']['arx_id'].to_list()
len(error_files)

In [ ]:
no_ror_df = res_df[pd.isna(res_df['name']) | pd.isna(res_df['ror'])]
no_ror_df.shape
no_ror_df.head()

In [ ]:
nn_df = res_df[pd.isna(res_df['name'])]
nn_df.shape
nn_df.head()

In [ ]:
res_list = []
for arx in tqdm(nn_df['arx_id'][:5]):
    res_list.append(phase_one.get_single_file_results(arx))

In [ ]:
res_list

In [134]:
importlib.reload(phase_one)

<module 'phase_one_json' from '/home/jupyter/metadata-vertexai/phase_one_json.py'>

### Examine extract

In [153]:
arx_id = '2311.03637v1'  #2311.03508v2
phase_one.get_single_file_results(arx_id, verbose=True, vverbose=True)

Processing ftp/arxiv/papers/2311/2311.03637.tar.gz
	Processing ftp/arxiv/papers/2311/2311.03637.tar.gz, main_v0.tex
0: \author[
{Jeremy Heyl$^{1}$,
Roberto Taverna$^{2}$,
Roberto Turolla$^{2,3}$,
Gian Luca Israel$^{4}$,
Mason Ng$^{5}$,
\newauthor
Demet Kirmizibayrak$^{1}$,
Denis Gonz\'alez-Caniulef$^{6}$,
Ilaria Caiazzo$^{7}$,
Silvia Zane$^{3}$,
Steven R. Ehlert$^{8}$,
\newauthor
Michela Negro$^{9}$,
Iv\'an Agudo$^{10}$,
Lucio Angelo Antonelli$^{4,11}$,
Matteo Bachetti$^{12}$,
Luca Baldini$^{13,14}$,
\newauthor
Wayne H. Baumgartner$^{8}$,
Ronaldo Bellazzini$^{13}$,
Stefano Bianchi$^{15}$,
Stephen D. Bongiorno$^{8}$,
\newauthor
Raffaella Bonino$^{16,17}$,
Alessandro Brez$^{13}$,
Niccol\`o Bucciantini$^{18,19,20}$,
Fiamma Capitanio$^{21}$,
\newauthor
Simone Castellano$^{13}$,
Elisabetta Cavazzuti$^{22}$,
Chien-Ting Chen$^{23}$,
Stefano Ciprini$^{24,11}$,
Enrico Costa$^{21}$,
\newauthor
Alessandra De Rosa$^{21}$,
Ettore Del Monte$^{21}$,
Laura Di Gesu$^{22}$,
Niccol\`o Di Lalla$^{25}$,
Al

[('2311.03637v1', 'University of British Columbia', 'Vancouver', '03rmrcq20'),
 ('2311.03637v1', 'Università degli Studi di Padova', 'Padova', '00240q980'),
 ('2311.03637v1',
  'University College London',
  'Holmbury St Mary',
  '02jx3x895'),
 ('2311.03637v1',
  'INAF Osservatorio Astronomico di Roma',
  'Monte Porzio Catone',
  '02hnp4676'),
 ('2311.03637v1',
  'Massachusetts Institute of Technology',
  'Cambridge',
  '042nb2s44'),
 ('2311.03637v1',
  'Institut de Recherche en Astrophysique et Planétologie',
  'Toulouse',
  '05hm2ja81'),
 ('2311.03637v1',
  'California Institute of Technology',
  'Pasadena',
  '05dxps055'),
 ('2311.03637v1',
  'NASA Marshall Space Flight Center',
  'Huntsville',
  '02epydz83'),
 ('2311.03637v1', 'Louisiana State University', 'Baton Rouge', '05ect4e57'),
 ('2311.03637v1',
  'Instituto de Astrofísica de Andalucía',
  'Granada',
  '04ka0vh05'),
 ('2311.03637v1', 'Agenzia Spaziale Italiana', 'Roma', '034zgem50'),
 ('2311.03637v1',
  'INAF Osservatorio As

In [ ]:
arx_id = '2311.03508v2'

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
tex_main = phase_one.find_main_tex_source_in_tar(tar_path)[0]

doc = """
Parses a .tex file:
- Removes LaTeX comments
- Extracts institution names (via recursive regex)
- Extracts text before the abstract
"""
from google.cloud import storage
PROJECT_ID = "arxiv-development"
PRD_PROJECT = 'arxiv-production'
PRD_BUCKET_LOC = 'arxiv-production-data' 

client = storage.Client(project=PRD_PROJECT)
bucket = client.bucket(PRD_BUCKET_LOC)
blob = bucket.blob(tar_path)
tar_bytes = blob.download_as_bytes()

if tar_path.endswith(".tar.gz"):
    try:
        with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode='r') as in_tar:
            tex_files = [f for f in in_tar.getnames() if f.endswith('.tex')]
            fp = in_tar.extractfile(tex_main)
            wrapped_file = io.TextIOWrapper(fp, newline=None, encoding='utf-8') #universal newlines
            source_text = phase_one.pre_format(wrapped_file.read())
    except UnicodeDecodeError:
        try:
            with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode='r') as in_tar:
                fp = in_tar.extractfile(tex_main)
                raw_data = in_tar.extractfile(tex_main).peek(10000)
                result = chardet.detect(raw_data)
                detected_encoding = result["encoding"]
                wrapped_file = io.TextIOWrapper(
                    fp, 
                    newline=None, 
                    encoding=detected_encoding, 
                    errors="replace"
                ) #universal newlines
                source_text = wrapped_file.read()
        except Exception as e:
            print(
                f"Failed to read {tar_path}-{tex_main} with"
                " detected encoding {detected_encoding}: {e}"
            )
            #return None
else:
    try:
        with gzip.open(io.BytesIO(tar_bytes), 'rt', encoding='utf-8') as in_gz:
            source_text = in_gz.read()
    except UnicodeDecodeError:
        try:
            with gzip.open(io.BytesIO(tar_bytes), 'rb') as in_gz:
                raw_data = in_gz.peek(10000)
                result = chardet.detect(raw_data)
                detected_encoding = result["encoding"]
            with gzip.open(
                io.BytesIO(tar_bytes),
                'rt', 
                encoding=detected_encoding
            ) as in_gz:
                source_text = in_gz.read()
        except Exception as e:
            print(
                f"Failed to read {tar_path}-{tex_main} with"
                " detected encoding {detected_encoding}: {e}"
            )

# Remove LaTeX comments (lines starting with non-escaped %)
content = re.sub(r"(?<!\\)%.*", "", source_text)
#res_list = []

# try parsing latex:
auth_macros = set([
    "author", "auth", "authors",
    "institute", "inst", "institution",
    "affiliation", "affil", "affiliations",
    "address",
    "cmsinstitute",
])
supstr = set([
    "\\textsuperscript",
])
latex_extracted_institutions = []
try:
    lxwkr = LatexWalker(content)
    (nodelist, pos, len_) = lxwkr.get_latex_nodes()
    focus_nodes = [
      (i,node) for i,node in enumerate(nodelist)
      if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
    ]
    if focus_nodes:
        for i,node in focus_nodes:
            latex_extracted_institutions.append(node.latex_verbatim())
            try:
                idx_plus = 1
                while True:
                    if idx_plus > 10:
                        break
                    follow_node = nodelist[i+idx_plus]
                    if isinstance(follow_node, LatexGroupNode):
                        latex_extracted_institutions.append(follow_node.latex_verbatim())
                        break
                    idx_plus += 1
            except IndexError:
                pass
        if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
            sup_res = extract_texsuperscript(nodelist)
            latex_extracted_institutions.extend(sup_res)
    else:
        doc = [
            node for node in nodelist
            if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document'
        ]
        if doc:
            docnodelist = doc[0].nodelist
            focus_doc_nodes = [
              (i,node) for i, node in enumerate(docnodelist)
              if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
            ]
            for i, node in focus_doc_nodes:
                latex_extracted_institutions.append(node.latex_verbatim())
                try:
                    idx_plus = 1
                    while True:
                        if idx_plus > 10:
                            break
                        follow_node = docnodelist[i+idx_plus]
                        if isinstance(follow_node, LatexGroupNode):
                            latex_extracted_institutions.append(follow_node.latex_verbatim())
                            break
                        idx_plus += 1
                except IndexError:
                    pass
            if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
                sup_res = extract_texsuperscript(doc[0].nodelist)
                latex_extracted_institutions.extend(sup_res)
    if latex_extracted_institutions:
        #res_list.append(latex_extracted_institutions)
        #yield "\n".join(latex_extracted_institutions)
        pass
except Exception as e:
    print(f"Overly broad except in extract_pre_abstract_content(): {e}")
    pass

#  "recursive" regex:
#   ((?>[^{}]+|\{(?1)\})*)
# optional brackets
#   (:?\[\d+\])?\s*
# This matches text possibly containing normal characters or nested braces,
# until the outermost braces are matched.
# If your LaTeX does not have deep nesting, this mainly ensures things like $^{1}$ are correctly parsed.
institution_patterns = [
    r"\\affiliation\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\institute\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\address\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\inst\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\affil\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\author\s*(:?\[\d+\])?\s*{[^}]+}{([^}]+)}",
    r"\\cmsinstitute\s*(:?\[\d+\])?\s*{[^}]+}{([^}]+)}",
]

extracted_institutions = []
for pattern in institution_patterns:
    # Use regex.findall with DOTALL to allow '.' to match newlines
    matches = re.findall(pattern, content, flags=re.DOTALL)
    if matches:
        # Strip each match and add to list
        for m in matches:
            if isinstance(m, tuple):
                extracted_institutions.append(" ".join(m_i for m_i in m))
            else:
                extracted_institutions.extend(m.strip() for m in matches if m.strip())

# If any institution info is extracted, return the deduplicated joined text
if extracted_institutions:
    # You can change the join method; here we join by newline and use set to deduplicate
    #return "\n".join(set(extracted_institutions))
    #res_list.append("\n".join(set(extracted_institutions)))
    #yield "\n".join(set(extracted_institutions))
    pass

# If no institution found, try extracting the text before the abstract
match = re.split(
    r"\\begin\s*{\s*abstract\s*}|\\s*\\section\s*{\s*Abstract\s*}",
    content,
    maxsplit=1,
    flags=re.IGNORECASE
)
if len(match) > 1:
    #return match[0].strip()
    #res_list.append(match[0].strip())
    #yield match[0].strip()
    pass

# If still not found, return the first 1/3 of the content as a fallback
content_length = len(content)
if content_length > 0:
    one_third_length = max(content_length//3, 2000)
    #return content[:one_third_length].strip()
    #res_list.append(content[:one_third_length].strip())
    #yield content[:one_third_length].strip()
    pass

# If still not found, return an empty string
#if res_list:
#  yield res_list
#else:
# yield ["",]



In [22]:
latex_extracted_institutions
extracted_institutions
content[:200]

['\\author*',
 '{\\fnm{Lulu} \\sur{Gong}}',
 '\\author[',
 '{\\fnm{Fabio} \\sur{Pasqualetti}}',
 '\\author[',
 '{\\fnm{Thomas} \\sur{Papouin}}',
 '\\author*',
 '{\\fnm{ShiNung} \\sur{Ching}}',
 '\\affil',
 '{\\orgdiv{Department of Electrical and Systems Engineering}, \\orgname{Washington University in St. Louis}, \\orgaddress{\\city{St. Louis}, \\postcode{63130}, \\state{MO}, \\country{USA}}}',
 '\\affil',
 '{\\orgdiv{Department of Mechanical Engineering}, \\orgname{University of California at Riverside}, \\orgaddress{\\city{Riverside}, \\postcode{92521}, \\state{CA}, \\country{USA}}}',
 '\\affil',
 '{\\orgdiv{Department of Neuroscience}, \\orgname{Washington University in St. Louis}, \\orgaddress{\\city{St. Louis}, \\postcode{63110}, \\state{MO}, \\country{USA}}}']

[]

'\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\\documentclass[pdflatex,sn-mathphys]{sn-jnl}\n\n\n\n\n\n\n\n\n\n\n\n\\newcommand{\\diag}{\\mathrm{diag}}\n\\newcommand{\\jac}[1]{D\\mkern-0.75mu{#1}}\n\\usepackage{comment}\n\\usepackage{titlesec}\n\n'

In [12]:
nodelist[:10]

[LatexCharsNode(parsing_state=<parsing state 139982773778032>, pos=0, len=25, chars='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'),
 LatexMacroNode(parsing_state=<parsing state 139982773778032>, pos=25, len=44, macroname='documentclass', nodeargd=ParsedMacroArgs(argspec='[{', argnlist=[LatexGroupNode(parsing_state=<parsing state 139982773778032>, pos=39, len=22, nodelist=[LatexCharsNode(parsing_state=<parsing state 139982773778032>, pos=40, len=20, chars='pdflatex,sn-mathphys')], delimiters=('[', ']')), LatexGroupNode(parsing_state=<parsing state 139982773778032>, pos=61, len=8, nodelist=[LatexCharsNode(parsing_state=<parsing state 139982773778032>, pos=62, len=6, chars='sn-jnl')], delimiters=('{', '}'))]), macro_post_space=''),
 LatexCharsNode(parsing_state=<parsing state 139982773778032>, pos=69, len=12, chars='\n\n\n\n\n\n\n\n\n\n\n\n'),
 LatexMacroNode(parsing_state=<parsing state 139982773778032>, pos=81, len=33, macroname='newcommand', nodeargd=ParsedMacroArgs(argspec='*{[

In [13]:
focus_doc_nodes

[(5,
  LatexMacroNode(parsing_state=<parsing state 139982773778032>, pos=796, len=8, macroname='author', nodeargd=ParsedMacroArgs(argspec='{', argnlist=[LatexCharsNode(parsing_state=<parsing state 139982773778032>, pos=803, len=1, chars='*')]), macro_post_space='')),
 (11,
  LatexMacroNode(parsing_state=<parsing state 139982773778032>, pos=854, len=8, macroname='author', nodeargd=ParsedMacroArgs(argspec='{', argnlist=[LatexCharsNode(parsing_state=<parsing state 139982773778032>, pos=861, len=1, chars='[')]), macro_post_space='')),
 (17,
  LatexMacroNode(parsing_state=<parsing state 139982773778032>, pos=925, len=8, macroname='author', nodeargd=ParsedMacroArgs(argspec='{', argnlist=[LatexCharsNode(parsing_state=<parsing state 139982773778032>, pos=932, len=1, chars='[')]), macro_post_space='')),
 (23,
  LatexMacroNode(parsing_state=<parsing state 139982773778032>, pos=996, len=8, macroname='author', nodeargd=ParsedMacroArgs(argspec='{', argnlist=[LatexCharsNode(parsing_state=<parsing st

In [14]:
latex_extracted_institutions

['\\author*',
 '{2023}',
 '\\author[',
 '{2023}',
 '\\author[',
 '{theorem}',
 '\\author*',
 '{Proposition}',
 '\\affil',
 '{example}',
 '\\affil',
 '{remark}',
 '\\affil',
 '{thmstylethree}']

In [20]:
nodelist[30]

LatexGroupNode(parsing_state=<parsing state 139982773778032>, pos=377, len=9, nodelist=[LatexCharsNode(parsing_state=<parsing state 139982773778032>, pos=378, len=7, chars='example')], delimiters=('{', '}'))

In [16]:
macro_node=nodelist[29]
macro_node
arg_list = macro_node.nodeargd.argnlist
arg_node = arg_list[0]
isinstance(arg_node, LatexCharsNode)

LatexMacroNode(parsing_state=<parsing state 139982773778032>, pos=366, len=11, macroname='newtheorem', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')

IndexError: list index out of range

In [ ]:
def get_arg_contents(node_list, macro_idx):
    macro_node = nodelist[macro_idx]
    arg_list = None
    
    try:
        arg_list = macro_node.nodeargd.argnlist
        if not arg_list or len(arg_list) > 1:
            return ""
        arg_node = arg_list[0]
        if 
    except AttributeError:
        pass
    
    
        

    
    

In [177]:
def extract_texsuperscript(latex_node_list, res=None):
    '''for each superscript, get the contents of the next LatexCharsNode'''
    bailout_macros = set(['abstract', 'subsection'])
    if res is None:
        res = []
    for i, node in enumerate(latex_node_list):
        sublist = []
        #print(type(node))
        try:
            if node.macroname=='textsuperscript':
                run_started = False
                text_list = []
                for nnode in latex_node_list[i:]:
                    #print(type(nnode))
                    if isinstance(nnode, LatexCharsNode):
                        text_list.append(nnode.latex_verbatim())
                        run_started = True
                    elif isinstance(nnode, LatexMacroNode):
                        if nnode.macroname == '&':
                            text_list.append('&')
                        elif run_started:
                            break
                    elif not isinstance(nnode, LatexCharsNode):
                        if run_started:
                            break
                if text_list:
                    res.append(" ".join(text_list))
        except AttributeError:
            pass
        if isinstance(node, LatexMacroNode):
            try: 
                if node.macroname in bailout_macros:
                    break
                sublist = node.nodeargd.argnlist
                #print(sublist)
            except AttributeError:
                pass
        if isinstance(node, (LatexGroupNode, LatexEnvironmentNode)):
            try:
                sublist = node.nodelist
                #print(sublist)
            except AttributeError:
                pass
        if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document':
            break
        if sublist:
            extract_texsuperscript(sublist, res)
    return res

In [162]:
for node in nodelist[139].nodeargd.argnlist[0].nodelist:
    print(node)
    print('\n')
    print('--')
    print('\n')

LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2703, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')


--


LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2719, len=6, nodelist=[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2720, len=4, macroname='dag', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')], delimiters=('{', '}'))


--


LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2725, len=21, chars='Corresponding authors')


--




In [163]:
extract_texsuperscript(nodelist[139:140])  #nodelist[138:150])

['Corresponding authors']

[LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2702, len=45, nodelist=[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2703, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2719, len=6, nodelist=[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2720, len=4, macroname='dag', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')], delimiters=('{', '}')), LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2725, len=21, chars='Corresponding authors')], delimiters=('{', '}'))]

In [167]:
extract_texsuperscript(nodelist[139:])

['Corresponding authors',
 'Molecular Robotics, Kerala, India; ',
 'School of Computing and Information, University of Pittsburgh, Pennsylvania, PA, USA; ',
 'Health Informatics ',
 'Department of Computer Science, Courant Institute of Mathematical Sciences, New York University, New York, NY, USA; ',
 'Institute for Next Generation Healthcare, Mount Sinai Health System, New York, NY, USA; ',
 'Department of Preventive Cardiology, DeBakey Heart ',
 'Hasso Plattner Institute for Digital Health, Icahn School of Medicine at Mount Sinai, New York, NY, USA; ',
 'Faculty of Medicine, Imperial College London, London, UK; ',
 'School of Engineering Medicine,  Texas A',
 'Department of Translational Medical Sciences, Center for Genomic and Precision Medicine, Texas A']

In [103]:
extra_pats = ["\\textsuperscript"]

[pat in l for l in latex_extracted_institutions for pat in extra_pats]

[True]

In [115]:
focus_nodes

[(137,
  LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=1902, len=794, macroname='author', nodeargd=ParsedMacroArgs(argspec='{', argnlist=[LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=1909, len=787, nodelist=[LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=1910, len=28, chars='\n    Sonish Sivarajkumar, MS'), LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=1938, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=1954, len=3, nodelist=[LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=1955, len=1, chars='1')], delimiters=('{', '}')), LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=1957, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexGroupNode(parsing_state=<parsing state 140674525854880>, 

In [111]:
nodelist[141].nodeargd.argnlist[0].nodelist

[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2756, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2772, len=3, nodelist=[LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2773, len=1, chars='1')], delimiters=('{', '}')),
 LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2775, len=35, chars='Molecular Robotics, Kerala, India; '),
 LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2810, len=2, macroname='\n', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2812, len=6, chars='      '),
 LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2818, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexGroupNode(parsing_state=<pars

## Test

In [ ]:
os.cpu_count()

In [ ]:
# importlib.reload(phase_one)

## times

```
Batch size    parallel workers    thread workers    time               n       sec/item        errors
   5             2                   5                                 100     
  10             2                   5                                 100     
  10             2                  10                                 100     
  
   5             3                   5                2 min 50s        100     1.70
  10             3                   5                2 min  8s        100     1.28
  10             3                  10                2 min 23s        100     1.43
  
  10             8                  10                3min 44s        1000      .22              0
  20             8                  10                3min 15s.       1000      .21              0       6414, 255; 6717, 296
  20             8                  20                3min 27s        1000      .21              0
  50             8                  25                3min 21s        1000      .20 
  
  25            12                  25                8min 1s         1000      .48
 100            12                  25                12min 39s       1000      .73 
 

10                   3                          5                    1 min 14s
```

In [ ]:
%%time

test_ids_df = pd.read_csv("gs://institutional-extract-scratch/reference/arx_ids/2311_ids.csv")
#test_ids_df.head()
ids_2311_all = test_ids_df["arx_id"].unique()


tt = TicToc()

input_ids = set(ids_2311_all)
save_name = "2311_db"
sample_size = "all"
batch_size = 20
parallel_workers = 8
thread_workers = 10

def worker(arx_id_list):
    res = phase_one.process_tex_files(arx_id_list, max_workers=thread_workers)
    return res

def run_phase_one_in_parallel(arx_id_batches):
    res_list = []
    total_len = sum(len(x) for x in arx_id_batches)
    with concurrent.futures.ProcessPoolExecutor(max_workers=parallel_workers) as executor:
        futures = [executor.submit(worker, arx_id_list) for arx_id_list in arx_id_batches]
        for future in as_completed(futures): #tqdm(as_completed(futures), total=len(futures)):
            res = future.result()
            res_list.extend(res)
#        res = [future.result() for future in concurrent.futures.as_completed(futures)]
#        for batch in res:
#            res_list.extend(batch)
    return res_list

def format_results(arxid_inst_ror_list):
    res_list = []
    for key, group in tqdm(itr.groupby(arxid_inst_ror_list, key=lambda x: x[0])):
        ror_inst = []
        for x in group:
            clean_name = x[1].split('.', 1)[-1].strip()
            ror = 'null'
            if len(x) == 3:
                ror = x[2].strip()
            inst = {
                'name':clean_name,
                'ror_id':ror
            }
            ror_inst.append(inst)
        arx_rec = {
            "arxiv_id": key.strip(),
            "institutions_with_ror": ror_inst,
        }
        res_list.append(arx_rec)
    return res_list



try:
    objects = []
    with open(f"checkpoints/{save_name}_{sample_size}.pkl", 'rb') as cp_fp:
        while True:
            try:
                obj = pickle.load(cp_fp)
                objects.append(obj)
            except EOFError:
                break
except FileNotFoundError as e:
    pass

known_ids = []
known_res = []
for obj in objects:
    known_ids.extend(x[0] for x in obj)
    known_res.extend(obj)

known_ids = set(known_ids)
input_ids = input_ids - known_ids
print(f"Found checkpoints for {len(known_ids)} nodes.")

if sample_size != "all":
    input_ids = input_ids[:sample_size]

#import concurrent.futures
#import phase_one


os.environ["TOKENIZERS_PARALLELISM"] = "false" 

batches = np.array_split(list(input_ids), len(input_ids)//batch_size)
tt.tic()
print(f"Start: {len(input_ids)} in {len(batches)} batches")
with open(f"checkpoints/{save_name}_{sample_size}.pkl", 'ab') as cp_fp:
    res = run_phase_one_in_parallel(batches, cp_fp)
tt.toc()
known_res.extend(res)
res_df = pd.DataFrame.from_records(known_res, columns=['arx_id', 'name', 'ror'])
res_df.to_csv(f"gs://institutional-extract-scratch/output/{save_name}_{sample_size}.csv.zip", index=False)
#res_list = format_results(res)
tt.toc()

In [ ]:
len(res)
sum( 1 for x in res if x[1] == 'error' )
sum( 1 for x in res if x[2] == 'null' )

In [ ]:
len(res)
sum( 1 for x in res if x[1] == 'error' )
sum( 1 for x in res if x[2] == 'null' )

In [ ]:
res_df = pd.DataFrame.from_records(res, columns=['arx_id', 'name', 'ror'])
res_df.to_csv(f"gs://institutional-extract-scratch/output/2311_scopus_{sample_size}.csv.zip", index=False)

In [ ]:
res

In [ ]:
%%time

tt = TicToc()

sample_size = 100
batch_size = 20
parallel_workers = 3
thread_workers = 20
#import concurrent.futures
#import phase_one

def worker(arx_id_list):
    res = phase_one.process_tex_files(arx_id_list, max_workers=thread_workers)
    return res

def run_phase_one_in_parallel(arx_id_batches):
    res_list = []
    total_len = sum(len(x) for x in arx_id_batches)
    with concurrent.futures.ProcessPoolExecutor(max_workers=parallel_workers) as executor:
        futures = [executor.submit(worker, arx_id_list) for arx_id_list in arx_id_batches]
        res = [future.result() for future in concurrent.futures.as_completed(futures)]
        for batch in res:
            res_list.extend(batch)
    return res_list

def run_phase_two_in_sequence(arxid_inst_list):
    res_list = []
    for key, group in tqdm(itr.groupby(arxid_inst_list, key=lambda x: x[0])):
        ror_inst = []
        for x in group:
            clean_name = x[1].split('.', 1)[-1].strip()
            ror = get_ror(clean_name)
            inst = {
                'name':clean_name,
                'ror_id':ror
            }
            ror_inst.append(inst)
        arx_rec = {
            "arxiv_id": key.strip(),
            "institutions_with_ror": ror_inst,
        }
        res_list.append(arx_rec)
    return res_list

os.environ["TOKENIZERS_PARALLELISM"] = "false" 
input_ids = scopus_all[:sample_size]
batches = np.array_split(input_ids, len(input_ids)//batch_size)
tt.tic()
print(f"Start Phase 1, {len(input_ids)} in {len(batches)} batches")
res = run_phase_one_in_parallel(batches)
tt.toc()
res[:5]
print("Start Phase 2")
res_list = run_phase_two_in_sequence(res)
tt.toc()
res_list[:5]

In [ ]:
%%time

tt = TicToc()

sample_size = 100
batch_size = 20
parallel_workers = 3
thread_workers = 20
#import concurrent.futures
#import phase_one

def worker(arx_id_list):
    res = phase_one.process_tex_files(arx_id_list, max_workers=thread_workers)
    print(len(res))
    res_list = run_phase_two_in_sequence(res)
    return res_list

def run_phase_one_in_parallel(arx_id_batches):
    res_list = []
    total_len = sum(len(x) for x in arx_id_batches)
    with concurrent.futures.ProcessPoolExecutor(max_workers=parallel_workers) as executor:
        futures = [executor.submit(worker, arx_id_list) for arx_id_list in arx_id_batches]
        res = [future.result() for future in concurrent.futures.as_completed(futures)]
        for batch in res:
            res_list.extend(batch)
    return res_list

def run_phase_two_in_sequence(arxid_inst_list):
    res_list = []
    for key, group in itr.groupby(arxid_inst_list, key=lambda x: x[0]):
        ror_inst = []
        for x in group:
            clean_name = x[1].split('.', 1)[-1].strip()
            ror = get_ror(clean_name)
            inst = {
                'name':clean_name,
                'ror_id':ror
            }
            ror_inst.append(inst)
        arx_rec = {
            "arxiv_id": key.strip(),
            "institutions_with_ror": ror_inst,
        }
        res_list.append(arx_rec)
    return res_list


os.environ["TOKENIZERS_PARALLELISM"] = "false" 
input_ids = scopus_all[:sample_size]
batches = np.array_split(input_ids, len(input_ids)//batch_size)
tt.tic()
print(f"Start Phase 1, {len(input_ids)} in {len(batches)} batches")
res = run_phase_one_in_parallel(batches)
tt.toc()
res[:5]
#print("Start Phase 2")
#res_list = run_phase_two_in_sequence(res)
#res_list[:5]

In [ ]:
res[:5]

In [ ]:
res_list

## ROR Index experiments

In [11]:
ror_gspath = 'gs://institutional-extract-scratch/reference/v1.63-2025-04-03-ror-data_schema_v2.json'
fs = gcsfs.GCSFileSystem()
with fs.open(ror_gspath, "r", encoding="utf-8") as f:
    ror_data = json.load(f)

In [15]:
#Locate withdrawn and successors
ror_dict = {e['id']:e for e in ror_data}
wd_succ_dict = {}
inactive_set = set([
    'withdrawn',
    'inactive',
])
withdrawn_ror = {
    e['id']: e['id']
    for e in ror_data 
    if e.get('status',"") in inactive_set
}
max_follows = 10
follow_count = 0
while len(withdrawn_ror) > 0:
    if follow_count > max_follows:
        break
    for wd_ror, sc_ror in tqdm(withdrawn_ror.items()):
        wd_entity = ror_dict[sc_ror]
        successor_rel = [
            r['id'] for r in wd_entity.get('relationships',[])
            if r['type'] == 'successor'
        ]
        if len(successor_rel) < 1:
            continue
        succ_ror = successor_rel[0]
        wd_succ_dict[wd_ror] = succ_ror
    withdrawn_ror = {
        wd_ror: sc_ror 
        for wd_ror, sc_ror in wd_succ_dict.items()
        if sc_ror in withdrawn_ror.keys()
    }
    follow_count += 1

  0%|          | 0/2091 [00:00<?, ?it/s]

  0%|          | 0/44 [00:00<?, ?it/s]

In [16]:
len(wd_succ_dict)

1689

In [17]:
wd_succ_dict.get('https://ror.org/04zrf7b53')

'https://ror.org/01qrts582'

In [9]:
for i,entry in tqdm(enumerate(ror_data)):
    ror_id = entry.get("id", "")
    if ror_id.endswith('04zrf7b53'):
        break

0it [00:00, ?it/s]

In [10]:
entry

{'locations': [{'geonames_id': 2894003,
   'geonames_details': {'continent_code': 'EU',
    'continent_name': 'Europe',
    'country_code': 'DE',
    'country_name': 'Germany',
    'country_subdivision_code': 'RP',
    'country_subdivision_name': 'Rheinland-Pfalz',
    'lat': 49.443,
    'lng': 7.77161,
    'name': 'Kaiserslautern'}}],
 'established': 1970,
 'external_ids': [{'type': 'fundref', 'all': ['100009044'], 'preferred': None},
  {'type': 'grid', 'all': ['grid.7645.0'], 'preferred': 'grid.7645.0'},
  {'type': 'isni', 'all': ['0000 0001 2155 0333'], 'preferred': None},
  {'type': 'wikidata', 'all': ['Q678965'], 'preferred': None}],
 'id': 'https://ror.org/04zrf7b53',
 'domains': [],
 'links': [{'type': 'website',
   'value': 'http://www.uni-kl.de/no_cache/en/home/'},
  {'type': 'wikipedia',
   'value': 'http://en.wikipedia.org/wiki/Kaiserslautern_University_of_Technology'}],
 'names': [{'value': 'Kaiserslautern University of Technology',
   'types': ['alias'],
   'lang': 'en'},


In [115]:
for i,entry in tqdm(enumerate(ror_data)):
    ror_id = entry.get("id", "")
    if ror_id.endswith('00jjx8s55'):
        break
entry

0it [00:00, ?it/s]

{'admin': {'created': {'date': '2018-11-14', 'schema_version': '1.0'},
  'last_modified': {'date': '2025-03-26', 'schema_version': '2.1'}},
 'domains': [],
 'established': 1945,
 'external_ids': [{'all': ['501100006489'],
   'preferred': None,
   'type': 'fundref'},
  {'all': ['grid.5583.b'], 'preferred': 'grid.5583.b', 'type': 'grid'},
  {'all': ['0000 0001 2299 8025'], 'preferred': None, 'type': 'isni'},
  {'all': ['Q868550'], 'preferred': None, 'type': 'wikidata'}],
 'id': 'https://ror.org/00jjx8s55',
 'links': [{'type': 'website', 'value': 'http://www.cea.fr/'},
  {'type': 'wikipedia',
   'value': 'https://en.wikipedia.org/wiki/French_Alternative_Energies_and_Atomic_Energy_Commission'}],
 'locations': [{'geonames_details': {'continent_code': 'EU',
    'continent_name': 'Europe',
    'country_code': 'FR',
    'country_name': 'France',
    'country_subdivision_code': 'IDF',
    'country_subdivision_name': 'Île-de-France',
    'lat': 48.85341,
    'lng': 2.3488,
    'name': 'Paris'},


## Scratch

In [104]:
arx_id = '2311.08680v2'

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
#tex_main = phase_one.find_main_tex_source_in_tar(tar_path)[0]

include_pat = re.compile(r'\\(?:input|include|subfile)\s*(?:\[.+\])?\s*\{([^}]+)\}')


doc = """
Parses a .tex file:
- Removes LaTeX comments
- Extracts institution names (via recursive regex)
- Extracts text before the abstract
"""
from google.cloud import storage
PROJECT_ID = "arxiv-development"
PRD_PROJECT = 'arxiv-production'
PRD_BUCKET_LOC = 'arxiv-production-data' 

client = storage.Client(project=PRD_PROJECT)
bucket = client.bucket(PRD_BUCKET_LOC)
blob = bucket.blob(tar_path)
tar_bytes = blob.download_as_bytes()

if tar_path.endswith(".tar.gz"):
        with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode='r') as in_tar:
            tex_files = [f for f in in_tar.getnames() if f.endswith('.tex')]
            print(tex_files)
            fp = in_tar.extractfile('main.tex')
            wrapped_file = io.TextIOWrapper(fp, newline=None, encoding='utf-8')
            [x for x in phase_one.find_included_files(wrapped_file) if x in tex_files]

['Acknowledgements-only.tex', 'GAL-only.tex', 'main.tex', 'SAL1.tex']


[]

In [84]:
include_pat = re.compile(r'\\(?:input|include|subfile)\s*(?:\[.+\])?\s*\{([^}]+)\}')

include_pat.findall(wrapped_file.read())

ValueError: I/O operation on closed file

In [72]:
text = r"""
\documentclass[twocolumn,twocolappendix, tighten]{aastex63}
\usepackage{graphicx}
\usepackage{url}
\usepackage{epstopdf}
\usepackage{color}
\usepackage{textcomp}
\usepackage{gensymb}
\usepackage{multirow}
\usepackage{ragged2e}
\usepackage{url}
\usepackage{amsmath} 
\usepackage{booktabs}

\usepackage{comment}
\usepackage{afterpage}
\usepackage{mathtools}
\usepackage{tabularx}
\usepackage{bold-extra}
\usepackage{xspace}
\usepackage{relsize}
\usepackage[utf8x]{inputenc} 
\usepackage{braket}

\usepackage[nolist,nohyperlinks]{acronym}

\usepackage{eht}

\usepackage[T1]{fontenc} 


\usepackage{comment}
\usepackage[encapsulated]{CJK}
\usepackage{ucs}
\usepackage[utf8x]{inputenc}
\newcommand{\cntext}[1]{\begin{CJK}{UTF8}{gbsn}#1\end{CJK}}



\newcommand{\sm}[1]{\textcolor{purple}{(SM: #1) }}
\newcommand{\dpm}[1]{\textcolor{red}{(DPM: #1) }}
\newcommand{\mdj}[1]{\textcolor{blue}{(MDJ: #1) }}
\newcommand{\edt}[1]{{\bf #1}}
\newcommand{\red}[1]{{\color{red} #1}}

\newcommand{\eri}[1]{\textcolor{magenta}{ER: #1}}


\DeclareGraphicsExtensions{.pdf,.png,.jpeg}


\interfootnotelinepenalty=10000

\begin{document}




\newcounter{iPap}\setcounter{iPap}{1}
\newcommand{\ehtsubtitle}{This is just the GAL for now}

\ifnum\value{iPap}=1 \renewcommand{\ehtsubtitle}{The Shadow of the Supermassive Black Hole in the Center of the Milky Way}\fi
\ifnum\value{iPap}=2 \renewcommand{\ehtsubtitle}{EHT and Multi-wavelength Observations, Data Processing, and Calibration}\fi
\ifnum\value{iPap}=3 \renewcommand{\ehtsubtitle}{Imaging of the Galactic Centre Supermassive Black Hole}\fi
\ifnum\value{iPap}=4 \renewcommand{\ehtsubtitle}{Variability, morphology, and black hole mass}\fi
\ifnum\value{iPap}=5 \renewcommand{\ehtsubtitle}{Testing Astrophysical Models of the Galactic Center Black Hole}\fi
\ifnum\value{iPap}=6 \renewcommand{\ehtsubtitle}{Testing the Black Hole Metric}\fi

\shorttitle{\ehtsubtitle}
\shortauthors{The EHT Collaboration et al.}

\title{
First Sagittarius A* Event Horizon Telescope Results.
\Roman{iPap}. \ehtsubtitle}

\include{./GAL-only}

\collaboration{0}{The Event Horizon Telescope Collaboration}

\ifnum\value{iPap}=1 \include{./SAL1}\fi 
\ifnum\value{iPap}=2 \include{./SAL2}\fi
\ifnum\value{iPap}=3 \include{./SAL3}\fi
\ifnum\value{iPap}=4 \include{./SAL4}\fi
\ifnum\value{iPap}=5 \include{./SAL5}\fi
\ifnum\value{iPap}=6 \include{./SAL6}\fi


\begin{acronym}
\acro{sn}[$S/N$]{signal-to-noise ratio}
\acro{vlbi}[VLBI]{very long baseline interferometry}
\acroplural{vlbi}[VLBI]{Very long baseline interferometry}
\acro{grmhd}[GRMHD]{general relativistic magnetohydrodynamics}
\acro{mhd}[MHD]{magnetohydrodynamics}
\acro{as}[as]{arcseconds}
\acro{agn}[AGN]{Active Galactic Nuclei}
\acro{llagn}[LLAGN]{low-luminosity AGN}
\acro{cena}[Cen~A]{Centaurus A}
\acro{sgra}[Sgr\,A*]{Sagittarius~A*}
\acro{agn}[AGN]{active galactic nuclei}
\acro{eht}[EHT]{Event Horizon Telescope}
\acro{bhc}[BHC]{\href{https://blackholecam.org}{BlackHoleCam}}
\acro{tanami}[TANAMI]{Tracking Active Galactic Nuclei with Austral Milliarcsecond Interferometry}
\acro{smbh}[SMBH]{supermassive black hole}
\acroplural{smbh}[SMBHs]{supermassive black holes}
\acro{aa}[ALMA]{Atacama Large Millimeter/submillimeter Array}
\acro{ap}[APEX]{Atacama Pathfinder Experiment}
\acro{pv}[PV]{IRAM~30\,m Telescope}
\acro{jc}[JCMT]{James Clerk Maxwell Telescope}
\acro{lm}[LMT]{Large Millimeter Telescope Alfonso Serrano}
\acro{sp}[SPT]{South Pole Telescope}
\acro{sm}[SMA]{Submillimeter Array}
\acro{az}[SMT]{Submillimeter Telescope}
\acro{chandra}[Chandra]{Chandra X-ray Observatory}
\acro{c}[c]{speed of light}
\acro{pc}[pc]{parsec}
\acro{gr}[GR]{general relativity}
\acro{aips}[\textsc{aips}]{\href{http://www.aips..edu}{Astronomical Image Processing System}}
\acro{casa}[CASA]{\href{https://casa..edu}{Common Astronomy Software Applications}}
\acro{symba}[\textsc{symba}]{\href{https://bitbucket.org/M_Janssen/symba}{SYnthetic Measurement creator for long Baseline Arrays}}
\acro{rpicard}[\textsc{Rpicard}]{\href{https://bitbucket.org/M_Janssen/picard}{Radboud PIpeline for the Calibration of high Angular Resolution Data}}
\acro{hops}[HOPS]{\href{https://www.haystack.mit.edu/tech/vlbi/hops.html}{Haystack Observatory Postprocessing System}}
\acro{hbt}[HBT]{Hanbury Brown and Twiss}
\acro{tov}[TOV]{Tolman-Oppenheimer-Volkoff}
\acro{mri}[MRI]{magnetorotational instability}
\acro{adaf}[ADAF]{advection-dominated accretion flow}
\acro{adios}[ADIOS]{adiabatic inflow-outflow solution}
\acro{cdaf}[CDAF]{convection-dominated accretion flow}
\acro{bz}[BZ]{Blandford-Znajek}
\acro{bp}[BP]{Blandford-Payne}
\acro{em}[EM]{electromagnetic}
\acro{blr}[BLR]{broad-line region}
\acro{nlr}[NLR]{narrow-line region}
\acro{ism}[ISM]{interstellar medium}
\acro{edf}[eDF]{electron distribution function}
\acro{pic}[PIC]{particle-in-cell}
\acro{sane}[SANE]{standard and normal evolution}
\acro{mad}[MAD]{magnetically arrested disk}
\acro{jive}[JIVE]{\href{http://www.jive.eu}{Joint Institute for VLBI ERIC}}
\acro{}[NRAO]{\href{https://www.nrao.edu}{National Radio Astronomy Observatory}}
\acro{muas}[$\mu$as]{microarcseconds}
\acroplural{agn}[AGN]{Active galactic nuclei}
\acro{jy}[Jy]{jansky}
\acro{pa}[PA]{position angle}
\acro{srmhd}[SRMHD]{special relativistic magnetohydrodynamics}
\acro{fov}[FOV]{field of view}
\acro{sefd}[SEFD]{system equivalent flux density}
\acroplural{sefd}[SEFDs]{system equivalent flux densities}
\end{acronym}
"""

include_pat = re.compile(r'\\(?:input|include|subfile)\s*(?:\[.+\])?\s*\{([^}]+)\}')


include_pat.findall(text)

['./GAL-only', './SAL1', './SAL2', './SAL3', './SAL4', './SAL5', './SAL6']

In [107]:
import string

In [115]:
np.lib.stride_tricks.sliding_window_view(np.array(list(string.ascii_letters)), window_shape=4)[::2]

array([['a', 'b', 'c', 'd'],
       ['c', 'd', 'e', 'f'],
       ['e', 'f', 'g', 'h'],
       ['g', 'h', 'i', 'j'],
       ['i', 'j', 'k', 'l'],
       ['k', 'l', 'm', 'n'],
       ['m', 'n', 'o', 'p'],
       ['o', 'p', 'q', 'r'],
       ['q', 'r', 's', 't'],
       ['s', 't', 'u', 'v'],
       ['u', 'v', 'w', 'x'],
       ['w', 'x', 'y', 'z'],
       ['y', 'z', 'A', 'B'],
       ['A', 'B', 'C', 'D'],
       ['C', 'D', 'E', 'F'],
       ['E', 'F', 'G', 'H'],
       ['G', 'H', 'I', 'J'],
       ['I', 'J', 'K', 'L'],
       ['K', 'L', 'M', 'N'],
       ['M', 'N', 'O', 'P'],
       ['O', 'P', 'Q', 'R'],
       ['Q', 'R', 'S', 'T'],
       ['S', 'T', 'U', 'V'],
       ['U', 'V', 'W', 'X'],
       ['W', 'X', 'Y', 'Z']], dtype='<U1')

In [126]:
width = 20000
overlap = 500
rng_srt = range(0, len(string.ascii_letters), width-overlap)
rng_stp = range(width, len(string.ascii_letters), width-overlap)

for i, j in itr.zip_longest(rng_srt, rng_stp):
    print(f"({i}, {j}) - {string.ascii_letters[i:j]}")
    if j is None: break

(0, 10) - abcdefghij
(7, 17) - hijklmnopq
(14, 24) - opqrstuvwx
(21, 31) - vwxyzABCDE
(28, 38) - CDEFGHIJKL
(35, 45) - JKLMNOPQRS
(42, None) - QRSTUVWXYZ
